# ASOC sparse index tracking — Notebook 1: initial tracker construction

This self-contained notebook constructs the initial sparse tracker used by the low-turnover maintenance notebook.

The defaults reproduce the paper-case branch when the same data are supplied and `RUN_PRODUCTION=True`.  The default `RUN_PRODUCTION=False` is a smoke-test mode for checking that the workflow runs quickly; smoke-mode numerical values are not expected to match the paper.

No LaTeX fragments are generated.  The notebook exports machine-readable CSV/JSON/NPZ outputs only.
## Reproducing the paper-case construction run

1. Place `merged_sp500_returns_2020_2025.csv` in `./data/`, or edit `DATA_PATH` in the configuration cell.
2. Set `RUN_PRODUCTION=True`.
3. Run the notebook top-to-bottom.
4. Confirm that the exported cache path ends in `initial_tracker_fit01_nnz150_c25_floor005_53names.npz`.
5. Use that cache as the input to Notebook 2.


In [ ]:
# ===============================================================
# GitHub configuration — initial tracker construction
# ===============================================================
# Data contract:
#   * CSV of daily returns, one row per date;
#   * first column is a parseable date index;
#   * asset-return columns plus one benchmark/index-return column;
#   * no missing values;
#   * fixed universe over the experiment.
#
# Paper reproduction defaults assume the supplied S&P 500-style dataset
# with asset columns first and the benchmark column named "SP500".
# ===============================================================

from pathlib import Path
import os

# Reproducibility: set BLAS/thread controls before NumPy/Pandas are imported.
for _thread_env_var in [
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    os.environ[_thread_env_var] = "1"


# Optional Colab Drive mount.  Leave False for local/Jupyter use.
MOUNT_GOOGLE_DRIVE = False
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print(f"Google Drive mount skipped: {exc}")

# User-editable paths.
DATA_PATH = Path("./data/merged_sp500_returns_2020_2025.csv")
OUTPUT_ROOT = Path("./outputs")

# Dataset format.  Set INDEX_COL="__LAST__" to use the final column.
INDEX_COL = "SP500"

# Paper-case default windows.  If the locked paper dates are unavailable,
# the notebook falls back to row-count windows using the two lengths below.
USE_PAPER_WINDOWS_IF_AVAILABLE = True
FIT_WINDOW_LENGTH = 500
HOLD_WINDOW_LENGTH = 63
ABSORB_FINAL_STUB = True

# Main user-facing construction settings.
NNZ_TARGET = 150
NNZ_EFF_THRESH = 1e-4
PRACTICAL_POS_EPS = 1e-3
PRACTICAL_POS_PROB = 0.60

# One-off implementability floor.  The default cache exported by this
# notebook is floor-applied, matching the paper-case maintenance run.
APPLY_INITIAL_FLOOR = True
INITIAL_FLOOR_WEIGHT = 0.005
INITIAL_FLOOR_REALLOC = "proportional_survivors"

# Run mode.  Smoke mode checks that the notebook works; production mode
# is the expensive paper-reproduction setting.
RUN_PRODUCTION = False
RUN_MODE = "production" if RUN_PRODUCTION else "smoke"
OUTPUT_DIR = OUTPUT_ROOT / RUN_MODE
RANDOM_SEED = 12345
VERBOSE = True

# Advanced paper-case defaults.
C_GRID_CONSTRUCTION = [1.0, 10.0, 25.0, 50.0, 75.0, 100.0, 125.0, 150.0, 200.0]
TAU_C = 2e-3
NNZ_SIGMA = 25.0
TE_HI_FRAC = 5.0
TE_LO_FRAC = 0.0
INITIAL_TE_REF = 1e-5

# Numerical settings: smoke vs production.
CONSTRUCTION_SAPG_N_ITER = 15_000 if RUN_PRODUCTION else 1_500
CONSTRUCTION_SAPG_BURN_PR = 4_000 if RUN_PRODUCTION else 400
CONSTRUCTION_FISTA_MAX_ITER = 4_000 if RUN_PRODUCTION else 2_000
CONSTRUCTION_FISTA_TOL = 1e-6
MALA_TUNE_STEPS = 2_000 if RUN_PRODUCTION else 500
CONSTRUCTION_MALA_N_BURN = 20_000 if RUN_PRODUCTION else 2_000
CONSTRUCTION_MALA_N_ITER = 450_000 if RUN_PRODUCTION else 10_000
CONSTRUCTION_MALA_THIN = 1
CONSTRUCTION_ESS_MIN_S = 400.0 if RUN_PRODUCTION else 25.0
CONSTRUCTION_ESS_MIN_TE = 1_000.0 if RUN_PRODUCTION else 50.0
CONSTRUCTION_MAX_LAG_ESS = 200 if RUN_PRODUCTION else 50

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN_MODE = {RUN_MODE}")
print(f"DATA_PATH = {DATA_PATH}")
print(f"OUTPUT_DIR = {OUTPUT_DIR.resolve()}")

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — rolling data preparation
# T=500 FIT windows, quarterly holds, final stub absorbed into Hold 16
#
# This block prepares every FIT window:
#   (i) centers y and R,
#   (ii) computes weighted-l1 alpha scales,
#   (iii) stores everything in rolling_fit_prepared.
#
# Convention:
#   - merged dataframe has asset-return columns first;
#   - index-return column is last, unless explicitly supplied.
# ===============================================================

import os
import warnings
import numpy as np
import pandas as pd
from typing import Optional, Dict, Any, List

# ---------------------------------------------------------------
# Runtime config
# ---------------------------------------------------------------

warnings.filterwarnings("ignore")

rng = np.random.default_rng(RANDOM_SEED)

# ---------------------------------------------------------------
# Data — merged returns, assets first and index return as last column
# ---------------------------------------------------------------

from pathlib import Path

DATA_PATH = Path(DATA_PATH)
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PATH_MERGED = str(DATA_PATH)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"DATA_PATH does not exist: {DATA_PATH}\n"
        "Edit the configuration cell before running the notebook."
    )

df_ret = pd.read_csv(
    DATA_PATH,
    index_col=0,
    parse_dates=True,
).sort_index()

# Basic data checks.  Missing values should be handled before using the notebook.
if not df_ret.index.is_monotonic_increasing:
    raise ValueError("The date index must be sorted increasingly.")
if df_ret.isna().any().any():
    raise ValueError("The return matrix contains missing values; clean or impute them before running.")


def split_X_y(
    df: pd.DataFrame,
    idx_col: Optional[str] = None,
):
    """
    Split merged returns dataframe into asset-return matrix R and
    index-return vector y.

    Parameters
    ----------
    df : pd.DataFrame
        Merged returns dataframe.
    idx_col : str, optional
        Name of the index-return column. If None, the final column is used.

    Returns
    -------
    R : np.ndarray, shape (T, p)
        Asset returns.
    y : np.ndarray, shape (T,)
        Index returns.
    idx_col : str
        Name of the index-return column.
    """
    if idx_col is None:
        idx_col = df.columns[-1]

    if idx_col not in df.columns:
        raise KeyError(f"INDEX_COL={idx_col!r} is not present in the dataframe columns.")

    y = df[idx_col].to_numpy(float).reshape(-1)
    R = df.drop(columns=[idx_col]).to_numpy(float)

    return R, y, idx_col


def slice_window(
    df: pd.DataFrame,
    start: str,
    end: str,
    idx_col: Optional[str] = None,
):
    """
    Slice a date window and return dates, R, and y.
    """
    out = df.loc[start:end].copy()

    R, y, _ = split_X_y(out, idx_col=idx_col)

    assert out.index.is_monotonic_increasing
    assert len(out) > 0
    assert not np.isnan(R).any()
    assert not np.isnan(y).any()

    return out.index, R, y


def center_data(
    y: np.ndarray,
    R: np.ndarray,
) -> Dict[str, np.ndarray | float]:
    """
    Center the index and asset returns over one FIT window.

    The centering constants are stored so that tracking error can later
    be evaluated on raw, uncentered returns using the corresponding
    un-centering correction.
    """
    y = np.asarray(y, dtype=float).reshape(-1)
    R = np.asarray(R, dtype=float)

    y_mu = float(y.mean())
    R_mu = R.mean(axis=0)

    y_c = y - y_mu
    R_c = R - R_mu

    return {
        "y_c": y_c,
        "R_c": R_c,
        "y_mu": y_mu,
        "R_mu": R_mu,
    }


def compute_alpha_scales(
    R_c: np.ndarray,
    eps: float = 1e-8,
) -> np.ndarray:
    """
    Compute weighted-l1 alpha scales for one centered FIT design matrix.

    alpha_j is proportional to ||R_{.j}||_2 / sqrt(T), then mean-normalized
    to one. This keeps the scale convention consistent across rolling windows.
    """
    R_c = np.asarray(R_c, dtype=float)
    T, _ = R_c.shape

    s = np.linalg.norm(R_c, axis=0) / max(np.sqrt(T), 1.0)
    s = np.clip(s, eps, None)
    s = s / s.mean()

    return s


# ---------------------------------------------------------------
# Rolling split definition
# ---------------------------------------------------------------

PAPER_ROLLING_WINDOWS = [
    {"k": 1, "fit_start": "2020-01-03", "fit_end": "2021-12-27", "hold_start": "2021-12-28", "hold_end": "2022-03-25"},
    {"k": 2, "fit_start": "2020-04-02", "fit_end": "2022-03-25", "hold_start": "2022-03-28", "hold_end": "2022-06-27"},
    {"k": 3, "fit_start": "2020-07-02", "fit_end": "2022-06-27", "hold_start": "2022-06-28", "hold_end": "2022-09-27"},
    {"k": 4, "fit_start": "2020-10-02", "fit_end": "2022-09-27", "hold_start": "2022-09-28", "hold_end": "2022-12-27"},
    {"k": 5, "fit_start": "2021-01-04", "fit_end": "2022-12-27", "hold_start": "2022-12-28", "hold_end": "2023-03-27"},
    {"k": 6, "fit_start": "2021-04-01", "fit_end": "2023-03-27", "hold_start": "2023-03-28", "hold_end": "2023-06-27"},
    {"k": 7, "fit_start": "2021-07-01", "fit_end": "2023-06-27", "hold_start": "2023-06-28", "hold_end": "2023-09-27"},
    {"k": 8, "fit_start": "2021-10-01", "fit_end": "2023-09-27", "hold_start": "2023-09-28", "hold_end": "2023-12-27"},
    {"k": 9, "fit_start": "2021-12-31", "fit_end": "2023-12-27", "hold_start": "2023-12-28", "hold_end": "2024-03-27"},
    {"k": 10, "fit_start": "2022-03-31", "fit_end": "2024-03-27", "hold_start": "2024-03-28", "hold_end": "2024-06-27"},
    {"k": 11, "fit_start": "2022-07-01", "fit_end": "2024-06-27", "hold_start": "2024-06-28", "hold_end": "2024-09-27"},
    {"k": 12, "fit_start": "2022-10-03", "fit_end": "2024-09-27", "hold_start": "2024-09-30", "hold_end": "2024-12-27"},
    {"k": 13, "fit_start": "2023-01-03", "fit_end": "2024-12-27", "hold_start": "2024-12-30", "hold_end": "2025-03-28"},
    {"k": 14, "fit_start": "2023-03-31", "fit_end": "2025-03-28", "hold_start": "2025-03-31", "hold_end": "2025-06-27"},
    {"k": 15, "fit_start": "2023-06-30", "fit_end": "2025-06-27", "hold_start": "2025-06-30", "hold_end": "2025-09-29"},
    {"k": 16, "fit_start": "2023-10-02", "fit_end": "2025-09-29", "hold_start": "2025-09-30", "hold_end": "2025-12-31"},
]

T_fit = int(FIT_WINDOW_LENGTH)


def _paper_windows_available(df: pd.DataFrame, windows: list[dict]) -> bool:
    """Return True when all locked paper-window boundary dates are in df.index."""
    idx = pd.DatetimeIndex(df.index)
    for win in windows:
        for key in ["fit_start", "fit_end", "hold_start", "hold_end"]:
            if pd.Timestamp(win[key]) not in idx:
                return False
    return True


def make_row_count_rolling_windows(
    df: pd.DataFrame,
    *,
    fit_length: int,
    hold_length: int,
    step_length: Optional[int] = None,
    absorb_final_stub: bool = True,
) -> list[dict]:
    """
    Create rolling FIT/HOLD windows by row counts.

    This is a generic fallback for non-paper datasets.  For exact paper
    reproduction with the supplied data, USE_PAPER_WINDOWS_IF_AVAILABLE=True
    uses the locked date windows above.
    """
    if fit_length <= 0 or hold_length <= 0:
        raise ValueError("fit_length and hold_length must be positive integers.")

    if step_length is None:
        step_length = hold_length
    if step_length <= 0:
        raise ValueError("step_length must be positive.")

    dates = pd.DatetimeIndex(df.index)
    windows = []
    k = 1
    fit_start_i = 0

    while True:
        fit_end_i = fit_start_i + fit_length - 1
        hold_start_i = fit_end_i + 1
        if hold_start_i >= len(dates):
            break

        hold_end_i = min(hold_start_i + hold_length - 1, len(dates) - 1)
        windows.append({
            "k": k,
            "fit_start": str(dates[fit_start_i].date()),
            "fit_end": str(dates[fit_end_i].date()),
            "hold_start": str(dates[hold_start_i].date()),
            "hold_end": str(dates[hold_end_i].date()),
        })

        if hold_end_i >= len(dates) - 1:
            break

        fit_start_i += step_length
        k += 1

    if absorb_final_stub and windows:
        windows[-1]["hold_end"] = str(dates[-1].date())

    return windows


if bool(USE_PAPER_WINDOWS_IF_AVAILABLE) and _paper_windows_available(df_ret, PAPER_ROLLING_WINDOWS):
    rolling_windows = [dict(win) for win in PAPER_ROLLING_WINDOWS]
    rolling_window_source = "locked_paper_windows"
else:
    rolling_windows = make_row_count_rolling_windows(
        df_ret,
        fit_length=int(FIT_WINDOW_LENGTH),
        hold_length=int(HOLD_WINDOW_LENGTH),
        step_length=int(globals().get("ROLL_STEP_LENGTH", HOLD_WINDOW_LENGTH)),
        absorb_final_stub=bool(ABSORB_FINAL_STUB),
    )
    rolling_window_source = "row_count_windows"

if len(rolling_windows) == 0:
    raise RuntimeError("No rolling windows could be constructed from the supplied dataset.")

# ---------------------------------------------------------------
# Old-code-style aliases for the first window
# ---------------------------------------------------------------

idx_col_config = None if INDEX_COL in [None, "", "__LAST__"] else str(INDEX_COL)
R_all, y_all, idx_name = split_X_y(df_ret, idx_col=idx_col_config)

fit_start, fit_end = (
    rolling_windows[0]["fit_start"],
    rolling_windows[0]["fit_end"],
)

hold1_start, hold1_end = (
    rolling_windows[0]["hold_start"],
    rolling_windows[0]["hold_end"],
)

dates_fit, R_fit, y_fit = slice_window(
    df_ret,
    fit_start,
    fit_end,
    idx_col=idx_name,
)

dates_h1, R_hold1, y_hold1 = slice_window(
    df_ret,
    hold1_start,
    hold1_end,
    idx_col=idx_name,
)

print(
    "Full:",
    df_ret.shape,
    "| Assets:",
    df_ret.shape[1] - 1,
    "| Index col:",
    idx_name,
)

print(
    "Date range:",
    df_ret.index.min().date(),
    "->",
    df_ret.index.max().date(),
)

print(f"Rolling windows: {len(rolling_windows)} (source={rolling_window_source})")
print("Initial FIT:", R_fit.shape, "| Hold1:", R_hold1.shape)

# ---------------------------------------------------------------
# Prepare every FIT window
# ---------------------------------------------------------------

rolling_fit_prepared: List[Dict[str, Any]] = []

for win in rolling_windows:
    k = int(win["k"])

    dates_fit_k, R_fit_k, y_fit_k = slice_window(
        df_ret,
        win["fit_start"],
        win["fit_end"],
        idx_col=idx_name,
    )

    # Safety check: every FIT window should have exactly T_fit rows.
    assert len(dates_fit_k) == T_fit, (
        f"Window {k}: expected {T_fit} FIT rows, got {len(dates_fit_k)}."
    )

    cent_k = center_data(y_fit_k, R_fit_k)

    y_c_k = cent_k["y_c"]
    R_c_k = cent_k["R_c"]
    y_mu_k = cent_k["y_mu"]
    R_mu_k = cent_k["R_mu"]

    alpha_k = compute_alpha_scales(R_c_k)

    rolling_fit_prepared.append(
        {
            "k": k,
            "fit_start": win["fit_start"],
            "fit_end": win["fit_end"],
            "hold_start": win["hold_start"],
            "hold_end": win["hold_end"],

            # Raw FIT data
            "dates_fit": dates_fit_k,
            "R_fit": R_fit_k,
            "y_fit": y_fit_k,

            # Centered FIT data
            "R_c": R_c_k,
            "y_c": y_c_k,
            "R_mu": R_mu_k,
            "y_mu": y_mu_k,

            # Weighted-l1 scales
            "alpha": alpha_k,
        }
    )

# ---------------------------------------------------------------
# First-window aliases used by downstream helper cells
# ---------------------------------------------------------------

first_fit = rolling_fit_prepared[0]

y_c = first_fit["y_c"]
R_c = first_fit["R_c"]
y_mu = first_fit["y_mu"]
R_mu = first_fit["R_mu"]
alpha = first_fit["alpha"]

T, p = R_c.shape

print(f"First FIT centered shape: T={T}, p={p}")
print(
    f"First FIT alpha weights: "
    f"min={alpha.min():.3f}, max={alpha.max():.3f}, mean={alpha.mean():.3f}"
)

# ---------------------------------------------------------------
# Compact sanity-check table for all FIT windows
# ---------------------------------------------------------------

fit_prep_summary = pd.DataFrame(
    [
        {
            "k": d["k"],
            "fit_start": d["fit_start"],
            "fit_end": d["fit_end"],
            "fit_n": len(d["dates_fit"]),
            "p": d["R_c"].shape[1],
            "alpha_min": float(d["alpha"].min()),
            "alpha_max": float(d["alpha"].max()),
            "alpha_mean": float(d["alpha"].mean()),
            "y_mean_raw": float(d["y_mu"]),
        }
        for d in rolling_fit_prepared
    ]
)

print(fit_prep_summary.to_string(index=False))

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — OLS residual variance
#
# This block estimates the classical residual variance for every
# centered FIT window prepared in rolling_fit_prepared.
#
# Important:
#   The OLS residual variance is used as a baseline/reference scale.
#   The actual likelihood variance used downstream will be calibrated
#   through sigma2_eff(c) = c * sigma2_ols.
# ===============================================================

import numpy as np
import pandas as pd
from typing import Dict, Any, List


def estimate_sigma2_ols(
    y_c: np.ndarray,
    R_c: np.ndarray,
) -> Dict[str, Any]:
    """
    Classical residual variance estimator for the centered regression

        y_c = R_c w + eps,    eps ~ N(0, sigma^2 I_T),

    using the unconstrained OLS fit.

    Important
    ---------
    This estimator is only used as a classical reference scale.
    In the revised ASOC pipeline, the likelihood variance / temperature
    is later treated as a calibration parameter via

        sigma2_eff(c) = c * sigma2_ols.

    Parameters
    ----------
    y_c : np.ndarray, shape (T,)
        Centered index returns.
    R_c : np.ndarray, shape (T, p)
        Centered asset returns.

    Returns
    -------
    out : dict
        Dictionary with keys:
        - 'w_ols'          : OLS weights, shape (p,)
        - 'residuals'      : residual vector y_c - R_c w_ols, shape (T,)
        - 'sigma2_hat'     : unbiased residual variance estimate
        - 'sigma_hat'      : square root of sigma2_hat
        - 'rss'            : residual sum of squares
        - 'resid_rmse'     : sqrt(rss / T), not dof-corrected
        - 'dof'            : degrees of freedom T - rank(R_c)
        - 'rank'           : numerical rank of R_c
        - 'singular_values': singular values returned by np.linalg.lstsq
    """
    y_c = np.asarray(y_c, dtype=float).reshape(-1)
    R_c = np.asarray(R_c, dtype=float)

    T, p = R_c.shape

    if y_c.shape[0] != T:
        raise ValueError(
            f"Dimension mismatch: len(y_c)={y_c.shape[0]} but R_c has T={T} rows."
        )

    # OLS fit.
    # rcond=None uses NumPy's default cutoff for small singular values.
    w_ols, _, rank, singular_values = np.linalg.lstsq(R_c, y_c, rcond=None)

    residuals = y_c - R_c @ w_ols

    # Use T - rank(R_c), which is the appropriate dof if R_c is rank deficient.
    dof = T - int(rank)

    if dof <= 0:
        raise ValueError(
            f"Non-positive degrees of freedom: T - rank(R_c) = {dof}. "
            "The classical OLS residual variance estimator is not defined."
        )

    rss = float(residuals @ residuals)

    sigma2_hat = float(rss / dof)
    sigma_hat = float(np.sqrt(sigma2_hat))
    resid_rmse = float(np.sqrt(rss / T))

    return {
        "w_ols": w_ols,
        "residuals": residuals,
        "sigma2_hat": sigma2_hat,
        "sigma_hat": sigma_hat,
        "rss": rss,
        "resid_rmse": resid_rmse,
        "dof": dof,
        "rank": int(rank),
        "singular_values": singular_values,
    }


def add_ols_variance_to_fit_windows(
    rolling_fit_prepared: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """
    Estimate OLS residual variance for every prepared FIT window.

    The function returns a new list of dictionaries, preserving the original
    rolling FIT data and adding:
      - 'ols_var'
      - 'sigma2_ols'
      - 'sigma_ols'
      - 'w_ols'
      - 'residuals_ols'
    """
    out: List[Dict[str, Any]] = []

    for d in rolling_fit_prepared:
        ols_var_k = estimate_sigma2_ols(
            y_c=d["y_c"],
            R_c=d["R_c"],
        )

        # Shallow copy is enough here because we only add new top-level keys.
        d_new = dict(d)

        d_new["ols_var"] = ols_var_k
        d_new["sigma2_ols"] = ols_var_k["sigma2_hat"]
        d_new["sigma_ols"] = ols_var_k["sigma_hat"]
        d_new["w_ols"] = ols_var_k["w_ols"]
        d_new["residuals_ols"] = ols_var_k["residuals"]

        out.append(d_new)

    return out


# ---------------------------------------------------------------
# Run OLS variance estimation for all FIT windows
# ---------------------------------------------------------------

rolling_fit_with_sigma = add_ols_variance_to_fit_windows(
    rolling_fit_prepared
)

# ---------------------------------------------------------------
# First-window aliases used by downstream helper cells
# ---------------------------------------------------------------

first_sigma = rolling_fit_with_sigma[0]

ols_var = first_sigma["ols_var"]
sigma2_ols = first_sigma["sigma2_ols"]
sigma_ols = first_sigma["sigma_ols"]
w_ols = first_sigma["w_ols"]
r_hat = first_sigma["residuals_ols"]

print(f"First FIT OLS variance estimate sigma^2_hat = {sigma2_ols:.4e}")
print(f"First FIT OLS sigma estimate sigma_hat = {sigma_ols:.4e}")
print(f"First FIT degrees of freedom T - rank(R_c) = {ols_var['dof']}")
print(f"First FIT numerical rank rank(R_c) = {ols_var['rank']}")

# ---------------------------------------------------------------
# Compact variance summary table for all FIT windows
# ---------------------------------------------------------------

sigma_summary = pd.DataFrame(
    [
        {
            "k": d["k"],
            "fit_start": d["fit_start"],
            "fit_end": d["fit_end"],
            "T": d["R_c"].shape[0],
            "p": d["R_c"].shape[1],
            "rank": d["ols_var"]["rank"],
            "dof": d["ols_var"]["dof"],
            "sigma2_ols": d["sigma2_ols"],
            "sigma_ols": d["sigma_ols"],
            "resid_rmse": d["ols_var"]["resid_rmse"],
            "rss": d["ols_var"]["rss"],
        }
        for d in rolling_fit_with_sigma
    ]
)

print(sigma_summary.to_string(index=False))

# ---------------------------------------------------------------
# Quick range checks across rolling FIT windows
# ---------------------------------------------------------------

print(
    "\nRolling OLS sigma^2 range:",
    f"{sigma_summary['sigma2_ols'].min():.4e}",
    "to",
    f"{sigma_summary['sigma2_ols'].max():.4e}",
)

print(
    "Rolling OLS sigma range:",
    f"{sigma_summary['sigma_ols'].min():.4e}",
    "to",
    f"{sigma_summary['sigma_ols'].max():.4e}",
)

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — variance-grid core helpers
#
# Assumes upstream object:
#   rolling_fit_with_sigma
#
# Each element should contain:
#   k, fit_start, fit_end,
#   R_fit, y_fit,
#   R_c, y_c, R_mu, y_mu,
#   alpha,
#   sigma2_ols, w_ols
# ===============================================================

import time
import numpy as np
import pandas as pd
from typing import Optional, Dict, Any, List, Sequence, Tuple


def check_fit_windows(
    fit_windows: List[Dict[str, Any]],
) -> None:
    """
    Lightweight sanity check for the rolling FIT-window objects.
    """
    required = [
        "k", "fit_start", "fit_end",
        "R_fit", "y_fit",
        "R_c", "y_c", "R_mu", "y_mu",
        "alpha",
        "sigma2_ols", "w_ols",
    ]

    for d in fit_windows:
        missing = [key for key in required if key not in d]
        if missing:
            raise KeyError(
                f"FIT window {d.get('k', '?')} is missing keys: {missing}"
            )

        R_c = np.asarray(d["R_c"])
        y_c = np.asarray(d["y_c"]).reshape(-1)
        alpha = np.asarray(d["alpha"]).reshape(-1)

        if R_c.shape[0] != y_c.shape[0]:
            raise ValueError(
                f"FIT window {d['k']}: R_c/y_c row mismatch."
            )

        if R_c.shape[1] != alpha.shape[0]:
            raise ValueError(
                f"FIT window {d['k']}: R_c/alpha dimension mismatch."
            )

        if float(d["sigma2_ols"]) <= 0.0:
            raise ValueError(
                f"FIT window {d['k']}: sigma2_ols must be positive."
            )


def te_rmse_raw_centered(
    y_raw: np.ndarray,
    R_raw: np.ndarray,
    w: np.ndarray,
    y_mu: float,
    R_mu: np.ndarray,
) -> float:
    """
    Tracking-error RMSE on raw returns using the centered-regression convention.

    The centered model is

        y_c approximately R_c w,

    where

        y_c = y_raw - y_mu,
        R_c = R_raw - R_mu.

    Therefore the fitted raw return is

        y_hat_raw = y_mu + (R_raw - R_mu) @ w.
    """
    y_raw = np.asarray(y_raw, dtype=float).reshape(-1)
    R_raw = np.asarray(R_raw, dtype=float)
    w = np.asarray(w, dtype=float).reshape(-1)
    R_mu = np.asarray(R_mu, dtype=float).reshape(-1)

    y_hat_raw = float(y_mu) + (R_raw - R_mu) @ w
    err = y_raw - y_hat_raw

    return float(np.sqrt(np.mean(err ** 2)))


def soft_threshold_weighted(
    x: np.ndarray,
    lam: float,
    alpha: np.ndarray,
) -> np.ndarray:
    """
    Weighted soft-thresholding.

        S_{lam alpha}(x)_j = sign(x_j) max(|x_j| - lam alpha_j, 0).
    """
    x = np.asarray(x, dtype=float)
    alpha = np.asarray(alpha, dtype=float)

    return np.sign(x) * np.maximum(np.abs(x) - lam * alpha, 0.0)


def grad_smooth_budget(
    y_c: np.ndarray,
    R_c: np.ndarray,
    w: np.ndarray,
    sigma2: float,
    Lambda: float,
) -> np.ndarray:
    """
    Gradient of the smooth objective component

        0.5/sigma2 ||y_c - R_c w||^2
        + Lambda (1^T w - 1)^2.
    """
    y_c = np.asarray(y_c, dtype=float).reshape(-1)
    R_c = np.asarray(R_c, dtype=float)
    w = np.asarray(w, dtype=float).reshape(-1)

    p = R_c.shape[1]
    one = np.ones(p)

    resid = R_c @ w - y_c

    grad_data = (R_c.T @ resid) / sigma2
    grad_budget = 2.0 * Lambda * (np.sum(w) - 1.0) * one

    return grad_data + grad_budget


def objective_map(
    y_c: np.ndarray,
    R_c: np.ndarray,
    w: np.ndarray,
    alpha: np.ndarray,
    theta: float,
    sigma2: float,
    Lambda: float,
) -> float:
    """
    Full MAP objective:

        0.5/sigma2 ||y_c - R_c w||^2
        + Lambda (1^T w - 1)^2
        + theta sum_j alpha_j |w_j|.
    """
    y_c = np.asarray(y_c, dtype=float).reshape(-1)
    R_c = np.asarray(R_c, dtype=float)
    w = np.asarray(w, dtype=float).reshape(-1)
    alpha = np.asarray(alpha, dtype=float).reshape(-1)

    resid = y_c - R_c @ w

    data_term = 0.5 / sigma2 * float(resid @ resid)
    budget_term = Lambda * float((np.sum(w) - 1.0) ** 2)
    l1_term = theta * float(np.sum(alpha * np.abs(w)))

    return float(data_term + budget_term + l1_term)


def _budget_hessian_matvec(
    v: np.ndarray,
    R_c: np.ndarray,
    sigma2: float,
    Lambda: float,
) -> np.ndarray:
    """
    Matrix-vector product with

        A = R_c^T R_c / sigma2 + 2 Lambda 11^T.
    """
    v = np.asarray(v, dtype=float).reshape(-1)
    R_c = np.asarray(R_c, dtype=float)

    data_part = (R_c.T @ (R_c @ v)) / sigma2
    budget_part = 2.0 * Lambda * np.sum(v) * np.ones_like(v)

    return data_part + budget_part


def power_iteration_budget(
    R_c: np.ndarray,
    sigma2: float,
    Lambda: float,
    max_iter: int = 200,
    tol: float = 1e-7,
    rng: Optional[np.random.Generator] = None,
) -> float:
    """
    Estimate the largest eigenvalue of

        R_c^T R_c / sigma2 + 2 Lambda 11^T.

    This is the Lipschitz constant of the smooth objective component.
    """
    R_c = np.asarray(R_c, dtype=float)
    p = R_c.shape[1]

    rng = np.random.default_rng() if rng is None else rng

    v = rng.standard_normal(p)
    v /= np.linalg.norm(v) + 1e-15

    prev_val = 0.0
    val = 0.0

    for _ in range(max_iter):
        Av = _budget_hessian_matvec(
            v=v,
            R_c=R_c,
            sigma2=sigma2,
            Lambda=Lambda,
        )

        val = float(np.dot(v, Av))

        norm_Av = np.linalg.norm(Av)
        if norm_Av <= 1e-15:
            break

        v_new = Av / norm_Av

        if abs(val - prev_val) <= tol * max(1.0, abs(val)):
            break

        v = v_new
        prev_val = val

    return float(val)


def compute_steps(
    R_c: np.ndarray,
    sigma2: float,
    tau_c: float = 2e-3,
    power_max_iter: int = 200,
    power_tol: float = 1e-7,
    rng: Optional[np.random.Generator] = None,
) -> Dict[str, float]:
    """
    Compute Lipschitz and step-size quantities for one effective variance.

    The soft-budget penalty is

        Lambda (1^T w - 1)^2,

    with

        Lambda = 1 / (2 tau_c^2).

    Returns
    -------
    steps : dict
        Dictionary containing:
        - L
        - lambda_MY
        - gamma_MYULA
        - fista_step
        - sigma2_used
        - Lambda
        - power_iters
        - power_tol
    """
    Lambda = 1.0 / (2.0 * tau_c ** 2)

    L = power_iteration_budget(
        R_c=R_c,
        sigma2=sigma2,
        Lambda=Lambda,
        max_iter=power_max_iter,
        tol=power_tol,
        rng=rng,
    )

    if L <= 0.0:
        raise ValueError(f"Non-positive Lipschitz estimate: L={L}")

    lambda_MY = 1.0 / L
    gamma_MYULA = 0.9 / (L + 1.0 / lambda_MY)
    fista_step = 1.0 / L

    return {
        "L": float(L),
        "lambda_MY": float(lambda_MY),
        "gamma_MYULA": float(gamma_MYULA),
        "fista_step": float(fista_step),
        "sigma2_used": float(sigma2),
        "Lambda": float(Lambda),
        "power_iters": int(power_max_iter),
        "power_tol": float(power_tol),
    }

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — SAPG and MAP solvers
#
# This block provides:
#   - theta0 heuristic from OLS / budget-centred LS vector;
#   - MYULA step for the smoothed weighted-l1 posterior;
#   - SAPG estimate of theta_EB for one fixed effective variance;
#   - FISTA MAP solver for one fixed effective variance and theta.
#
# Assumes Block 1 has defined:
#   soft_threshold_weighted
#   grad_smooth_budget
#   objective_map
# ===============================================================


def theta0_weighted_ols_l1(
    y_c: np.ndarray,
    R_c: np.ndarray,
    alpha: np.ndarray,
    w_ols: Optional[np.ndarray] = None,
) -> Tuple[float, np.ndarray]:
    """
    Heuristic initial theta for the weighted-l1 prior.

    Uses an OLS vector, budget-centres it to sum to one, and sets

        theta0 = p / sum_j alpha_j |w_ls,j|.

    Parameters
    ----------
    y_c : np.ndarray, shape (T,)
        Centered index returns.
    R_c : np.ndarray, shape (T, p)
        Centered asset returns.
    alpha : np.ndarray, shape (p,)
        Weighted-l1 scale factors.
    w_ols : np.ndarray, optional
        Precomputed OLS vector. If omitted, it is recomputed.

    Returns
    -------
    theta0 : float
        Initial theta scale for SAPG.
    w_ls : np.ndarray, shape (p,)
        Budget-centred OLS vector used in the heuristic.
    """
    y_c = np.asarray(y_c, dtype=float).reshape(-1)
    R_c = np.asarray(R_c, dtype=float)
    alpha = np.asarray(alpha, dtype=float).reshape(-1)

    p = R_c.shape[1]

    if w_ols is None:
        w_ls = np.linalg.lstsq(R_c, y_c, rcond=None)[0].reshape(-1)
    else:
        w_ls = np.asarray(w_ols, dtype=float).reshape(-1)

    # Budget-centre the OLS vector so that sum(w_ls) = 1.
    w_ls = w_ls - (np.sum(w_ls) - 1.0) * np.ones(p) / p

    S_ls = float(np.sum(alpha * np.abs(w_ls)))
    theta0 = max(p / max(S_ls, 1e-8), 1e-6)

    return float(theta0), w_ls


def myula_step(
    w: np.ndarray,
    y_c: np.ndarray,
    R_c: np.ndarray,
    alpha: np.ndarray,
    theta: float,
    steps: Dict[str, float],
    rng: np.random.Generator,
) -> np.ndarray:
    """
    One MYULA step for the smoothed weighted-l1 posterior.

    The nonsmooth weighted-l1 term is handled through the Moreau--Yosida
    approximation using weighted soft-thresholding.
    """
    prox = soft_threshold_weighted(
        x=w,
        lam=steps["lambda_MY"] * theta,
        alpha=alpha,
    )

    grad = grad_smooth_budget(
        y_c=y_c,
        R_c=R_c,
        w=w,
        sigma2=steps["sigma2_used"],
        Lambda=steps["Lambda"],
    )

    drift = -grad - (w - prox) / steps["lambda_MY"]
    noise = np.sqrt(2.0 * steps["gamma_MYULA"]) * rng.normal(size=w.shape)

    return w + steps["gamma_MYULA"] * drift + noise


def smoothed_logpost(
    w: np.ndarray,
    y_c: np.ndarray,
    R_c: np.ndarray,
    alpha: np.ndarray,
    theta: float,
    steps: Dict[str, float],
) -> Tuple[float, float, float]:
    """
    Smoothed log-posterior up to an additive constant.

    Returns
    -------
    logpost : float
        Negative smoothed objective.
    S_prox : float
        Weighted-l1 size of the prox point.
    data_term : float
        Data plus soft-budget contribution.
    """
    resid = y_c - R_c @ w

    data_term = (
        0.5 / steps["sigma2_used"] * float(resid @ resid)
        + steps["Lambda"] * float((np.sum(w) - 1.0) ** 2)
    )

    prox = soft_threshold_weighted(
        x=w,
        lam=steps["lambda_MY"] * theta,
        alpha=alpha,
    )

    S_prox = float(np.sum(alpha * np.abs(prox)))

    g_smoothed = (
        0.5 / steps["lambda_MY"] * float(np.sum((w - prox) ** 2))
        + theta * S_prox
    )

    logpost = float(-(data_term + g_smoothed))

    return logpost, S_prox, float(data_term)


def run_sapg_theta(
    y_c: np.ndarray,
    R_c: np.ndarray,
    alpha: np.ndarray,
    theta0: float,
    theta_bounds: Tuple[float, float],
    steps: Dict[str, float],
    n_iter: int = 15_000,
    burn_pr: int = 4_000,
    c_sapg: float = 1.0,
    k0: int = 200,
    pr_q: float = 1.0,
    rng: Optional[np.random.Generator] = None,
) -> Dict[str, Any]:
    """
    SAPG for theta, written in eta = log(theta).

    Update:

        eta_{k+1}
        = projection[ eta_k + rho_k (p - theta_k S_k) ],

    where

        S_k = sum_j alpha_j |w_k,j|.

    Since the budget constraint is soft here, the score uses p, matching
    the old FIT-window code.
    """
    rng = np.random.default_rng() if rng is None else rng

    y_c = np.asarray(y_c, dtype=float).reshape(-1)
    R_c = np.asarray(R_c, dtype=float)
    alpha = np.asarray(alpha, dtype=float).reshape(-1)

    p = R_c.shape[1]

    w = np.zeros(p)
    eta = float(np.log(theta0))

    log_lo = float(np.log(theta_bounds[0]))
    log_hi = float(np.log(theta_bounds[1]))

    theta = float(np.exp(eta))

    eta_trace = np.empty(n_iter + 1)
    theta_trace = np.empty(n_iter + 1)
    S_trace = np.empty(n_iter + 1)
    data_term_trace = np.empty(n_iter + 1)
    logpost_trace = np.empty(n_iter + 1)

    logpost0, S0, data0 = smoothed_logpost(
        w=w,
        y_c=y_c,
        R_c=R_c,
        alpha=alpha,
        theta=theta,
        steps=steps,
    )

    eta_trace[0] = eta
    theta_trace[0] = theta
    S_trace[0] = S0
    data_term_trace[0] = data0
    logpost_trace[0] = logpost0

    for it in range(1, n_iter + 1):
        w = myula_step(
            w=w,
            y_c=y_c,
            R_c=R_c,
            alpha=alpha,
            theta=theta,
            steps=steps,
            rng=rng,
        )

        S = float(np.sum(alpha * np.abs(w)))

        rho = c_sapg / (it + k0)
        eta = float(np.clip(eta + rho * (p - theta * S), log_lo, log_hi))
        theta = float(np.exp(eta))

        logpost, _, data_term = smoothed_logpost(
            w=w,
            y_c=y_c,
            R_c=R_c,
            alpha=alpha,
            theta=theta,
            steps=steps,
        )

        eta_trace[it] = eta
        theta_trace[it] = theta
        S_trace[it] = S
        data_term_trace[it] = data_term
        logpost_trace[it] = logpost

    # Polyak--Ruppert average over eta.
    start = min(burn_pr + 1, n_iter)

    idx = np.arange(start, n_iter + 1, dtype=float)

    if len(idx) > 0:
        weights = (idx - burn_pr) ** pr_q
        eta_bar = float(np.sum(weights * eta_trace[start:]) / np.sum(weights))
    else:
        eta_bar = float(np.mean(eta_trace[1:]))

    theta_EB = float(np.exp(eta_bar))

    return {
        "eta_trace": eta_trace,
        "theta_trace": theta_trace,
        "S_trace": S_trace,
        "data_term_trace": data_term_trace,
        "logpost_trace": logpost_trace,
        "eta_bar": eta_bar,
        "theta_EB": theta_EB,
    }


def fista_map(
    y_c: np.ndarray,
    R_c: np.ndarray,
    alpha: np.ndarray,
    theta: float,
    steps: Dict[str, float],
    max_iter: int = 4_000,
    tol: float = 1e-6,
    nnz_eff_thresh: float = 1e-4,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    """
    FISTA for the MAP problem:

        min_w  0.5/sigma2 ||y_c - R_c w||^2
              + Lambda (1^T w - 1)^2
              + theta sum_j alpha_j |w_j|.

    The budget is handled as a soft quadratic penalty, matching the
    earlier workflow.
    """
    y_c = np.asarray(y_c, dtype=float).reshape(-1)
    R_c = np.asarray(R_c, dtype=float)
    alpha = np.asarray(alpha, dtype=float).reshape(-1)

    p = R_c.shape[1]

    w = np.zeros(p)
    z = w.copy()
    t = 1.0

    step = steps["fista_step"]

    obj_trace = []
    nnz_trace = []
    nnz_eff_trace = []
    S1_trace = []
    sum_w_trace = []

    prev_obj = np.inf
    k_last = max_iter

    t0 = time.perf_counter()

    for it in range(1, max_iter + 1):
        grad = grad_smooth_budget(
            y_c=y_c,
            R_c=R_c,
            w=z,
            sigma2=steps["sigma2_used"],
            Lambda=steps["Lambda"],
        )

        w_next = soft_threshold_weighted(
            x=z - step * grad,
            lam=step * theta,
            alpha=alpha,
        )

        t_next = 0.5 * (1.0 + np.sqrt(1.0 + 4.0 * t * t))
        z = w_next + ((t - 1.0) / t_next) * (w_next - w)

        obj = objective_map(
            y_c=y_c,
            R_c=R_c,
            w=w_next,
            alpha=alpha,
            theta=theta,
            sigma2=steps["sigma2_used"],
            Lambda=steps["Lambda"],
        )

        obj_trace.append(obj)
        nnz_trace.append(int(np.count_nonzero(w_next)))
        nnz_eff_trace.append(int(np.sum(np.abs(w_next) >= nnz_eff_thresh)))
        S1_trace.append(float(np.sum(alpha * np.abs(w_next))))
        sum_w_trace.append(float(np.sum(w_next)))

        if abs(prev_obj - obj) <= tol * max(1.0, abs(obj)):
            w = w_next
            k_last = it
            break

        w = w_next
        t = t_next
        prev_obj = obj

    trace = {
        "obj": np.asarray(obj_trace),
        "nnz": np.asarray(nnz_trace),
        "nnz_eff": np.asarray(nnz_eff_trace),
        "S1": np.asarray(S1_trace),
        "sum_w": np.asarray(sum_w_trace),
        "iters": int(k_last),
        "wall_seconds": float(time.perf_counter() - t0),
    }

    return w, trace

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — variance-grid interface
#
# Public functions:
#   run_fit_grid(...)
#   run_all_fit_grids(...)
#
# Rolling target convention:
#   FIT 1:
#       nnz_target = initial_nnz_target
#       TE_previous_ref = max(initial_te_ref, TE_OLS_on_this_FIT)
#
#   FIT k >= 2:
#       nnz_target = nnz(previous selected portfolio)
#       TE_previous_ref = TE(previous selected portfolio on current FIT window)
#
# The selected MAP from FIT k is provisionally treated as the held portfolio
# for setting the target in FIT k+1. Later, once UQ/gating is inserted,
# this can be replaced by the gated/rebalanced portfolio.
# ===============================================================


def count_effective_nnz(
    w: np.ndarray,
    thresh: float = 1e-4,
) -> int:
    """
    Count effective nonzeros.

    If thresh > 0, count entries with |w_j| >= thresh.
    If thresh <= 0, use exact np.count_nonzero.

    Using the same threshold for targets and reported nnz_eff avoids
    treating tiny numerical entries as economically meaningful holdings.
    """
    w = np.asarray(w, dtype=float).reshape(-1)

    if thresh <= 0.0:
        return int(np.count_nonzero(w))

    return int(np.sum(np.abs(w) >= thresh))


def score_grid_row(
    te_fit: float,
    te_ref: float,
    nnz_eff: int,
    nnz_target: int,
    te_hi_frac: float = 5.0,
    te_lo_frac: float = 0.0,
    nnz_sigma: float = 25.0,
) -> Tuple[float, float, float]:
    """
    TE/sparsity score for one c-grid row.

    TE gate:
        phi_TE = 1 if
            te_lo_frac * te_ref <= TE_fit <= te_hi_frac * te_ref,
        and 0 otherwise.

    Sparsity preference:
        w_nnz = exp(-0.5 * ((nnz_eff - nnz_target) / nnz_sigma)^2).

    Final score:
        score = phi_TE * w_nnz.

    Notes
    -----
    The default TE gate is intentionally loose because this is the
    pre-gating portfolio-building stage. We do not want to overfit the
    in-sample period before pruning and UQ/gating are applied.
    """
    te_ref = float(te_ref)
    te_fit = float(te_fit)

    lo = te_lo_frac * te_ref
    hi = te_hi_frac * te_ref

    phi_TE = 1.0 if (lo <= te_fit <= hi) else 0.0

    if nnz_sigma <= 0:
        w_nnz = 1.0
    else:
        w_nnz = float(
            np.exp(-0.5 * ((nnz_eff - nnz_target) / nnz_sigma) ** 2)
        )

    score = float(phi_TE * w_nnz)

    return phi_TE, w_nnz, score


def make_grid_summary(
    rows: List[Dict[str, Any]],
) -> pd.DataFrame:
    """
    Lightweight table from full grid rows.
    Excludes arrays and traces.
    """
    keep_cols = [
        "k",
        "fit_start",
        "fit_end",
        "c",
        "sigma2_base",
        "sigma2_eff",
        "theta0",
        "theta_EB",
        "L",
        "lambda_MY",
        "gamma_MYULA",
        "TE_fit",
        "TE_previous_ref",
        "TE_ref_OLS",
        "TE_ratio_vs_previous",
        "TE_ratio_vs_OLS",
        "nnz_target",
        "nnz_raw",
        "nnz_eff",
        "weighted_l1",
        "sum_w",
        "budget_error",
        "obj_final",
        "phi_TE",
        "w_nnz",
        "score",
        "sapg_seconds",
        "fista_seconds",
        "fista_iters",
    ]

    return pd.DataFrame([{key: row[key] for key in keep_cols} for row in rows])


def select_best_grid_row(
    rows: List[Dict[str, Any]],
    fallback: str = "closest_TE_ref",
) -> Dict[str, Any]:
    """
    Select best c-row.

    Primary rule:
        maximise score.

    Fallback if all scores are zero:
        - 'closest_TE_ref':
              choose TE_fit closest to TE_previous_ref;
        - 'min_TE':
              choose smallest FIT tracking error;
        - 'min_TE_ratio':
              choose smallest TE_fit / TE_ref_OLS ratio.
    """
    scores = np.asarray([row["score"] for row in rows], dtype=float)

    if np.any(scores > 0):
        best_idx = int(np.argmax(scores))
        return rows[best_idx]

    if fallback == "closest_TE_ref":
        best_idx = int(
            np.argmin(
                [
                    abs(row["TE_fit"] - row["TE_previous_ref"])
                    for row in rows
                ]
            )
        )
    elif fallback == "min_TE":
        best_idx = int(np.argmin([row["TE_fit"] for row in rows]))
    elif fallback == "min_TE_ratio":
        best_idx = int(np.argmin([row["TE_ratio_vs_OLS"] for row in rows]))
    else:
        raise ValueError(f"Unknown fallback rule: {fallback}")

    best = dict(rows[best_idx])
    best["selection_warning"] = "All scores were zero; fallback rule was used."

    return best


def run_one_c(
    fit_window: Dict[str, Any],
    c: float,
    theta0: float,
    theta_bounds: Tuple[float, float],
    te_previous_ref: float,
    nnz_target: int,
    tau_c: float = 2e-3,
    sapg_n_iter: int = 15_000,
    sapg_burn_pr: int = 4_000,
    sapg_c: float = 1.0,
    sapg_k0: int = 200,
    sapg_pr_q: float = 1.0,
    fista_max_iter: int = 4_000,
    fista_tol: float = 1e-6,
    nnz_eff_thresh: float = 1e-4,
    nnz_sigma: float = 25.0,
    te_hi_frac: float = 5.0,
    te_lo_frac: float = 0.0,
    power_max_iter: int = 200,
    power_tol: float = 1e-7,
    rng: Optional[np.random.Generator] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Run one c-value for one FIT window.

    sigma2_eff = c * sigma2_ols.

    For this sigma2_eff:
      1. recompute steps;
      2. run SAPG for theta_EB;
      3. solve MAP;
      4. compute TE/sparsity metrics;
      5. score against TE_previous_ref and nnz_target.
    """
    rng = np.random.default_rng() if rng is None else rng

    k = int(fit_window["k"])

    y_c = fit_window["y_c"]
    R_c = fit_window["R_c"]
    alpha = fit_window["alpha"]

    y_raw = fit_window["y_fit"]
    R_raw = fit_window["R_fit"]
    y_mu = fit_window["y_mu"]
    R_mu = fit_window["R_mu"]

    sigma2_base = float(fit_window["sigma2_ols"])
    sigma2_eff = float(c) * sigma2_base

    steps = compute_steps(
        R_c=R_c,
        sigma2=sigma2_eff,
        tau_c=tau_c,
        power_max_iter=power_max_iter,
        power_tol=power_tol,
        rng=rng,
    )

    t_sapg0 = time.perf_counter()

    sapg_out = run_sapg_theta(
        y_c=y_c,
        R_c=R_c,
        alpha=alpha,
        theta0=theta0,
        theta_bounds=theta_bounds,
        steps=steps,
        n_iter=sapg_n_iter,
        burn_pr=sapg_burn_pr,
        c_sapg=sapg_c,
        k0=sapg_k0,
        pr_q=sapg_pr_q,
        rng=rng,
    )

    sapg_seconds = float(time.perf_counter() - t_sapg0)

    theta_EB = float(sapg_out["theta_EB"])

    w_map, fista_trace = fista_map(
        y_c=y_c,
        R_c=R_c,
        alpha=alpha,
        theta=theta_EB,
        steps=steps,
        max_iter=fista_max_iter,
        tol=fista_tol,
        nnz_eff_thresh=nnz_eff_thresh,
    )

    te_ref_ols = te_rmse_raw_centered(
        y_raw=y_raw,
        R_raw=R_raw,
        w=fit_window["w_ols"],
        y_mu=y_mu,
        R_mu=R_mu,
    )

    te_fit = te_rmse_raw_centered(
        y_raw=y_raw,
        R_raw=R_raw,
        w=w_map,
        y_mu=y_mu,
        R_mu=R_mu,
    )

    abs_w = np.abs(w_map)

    nnz_raw = int(np.count_nonzero(w_map))
    nnz_eff = count_effective_nnz(w_map, thresh=nnz_eff_thresh)

    weighted_l1 = float(np.sum(alpha * abs_w))
    sum_w = float(np.sum(w_map))
    budget_error = float(sum_w - 1.0)

    obj_final = objective_map(
        y_c=y_c,
        R_c=R_c,
        w=w_map,
        alpha=alpha,
        theta=theta_EB,
        sigma2=sigma2_eff,
        Lambda=steps["Lambda"],
    )

    phi_TE, w_nnz, score = score_grid_row(
        te_fit=te_fit,
        te_ref=te_previous_ref,
        nnz_eff=nnz_eff,
        nnz_target=nnz_target,
        te_hi_frac=te_hi_frac,
        te_lo_frac=te_lo_frac,
        nnz_sigma=nnz_sigma,
    )

    row = {
        "k": k,
        "fit_start": fit_window["fit_start"],
        "fit_end": fit_window["fit_end"],

        "c": float(c),
        "sigma2_base": sigma2_base,
        "sigma2_eff": sigma2_eff,

        "theta0": float(theta0),
        "theta_EB": theta_EB,

        "L": steps["L"],
        "lambda_MY": steps["lambda_MY"],
        "gamma_MYULA": steps["gamma_MYULA"],

        "TE_fit": te_fit,
        "TE_previous_ref": float(te_previous_ref),
        "TE_ref_OLS": te_ref_ols,
        "TE_ratio_vs_previous": te_fit / max(float(te_previous_ref), 1e-16),
        "TE_ratio_vs_OLS": te_fit / max(te_ref_ols, 1e-16),

        "nnz_target": int(nnz_target),
        "nnz_raw": nnz_raw,
        "nnz_eff": nnz_eff,
        "weighted_l1": weighted_l1,
        "sum_w": sum_w,
        "budget_error": budget_error,
        "obj_final": obj_final,

        "phi_TE": phi_TE,
        "w_nnz": w_nnz,
        "score": score,

        "sapg_seconds": sapg_seconds,
        "fista_seconds": fista_trace["wall_seconds"],
        "fista_iters": fista_trace["iters"],

        "w_map": w_map,
        "steps": steps,
        "sapg_out": sapg_out,
        "fista_trace": fista_trace,
    }

    if verbose:
        print(
            f"[FIT-{k:02d}] "
            f"c={float(c):7.3g} | "
            f"sigma2={sigma2_eff:.3e} | "
            f"theta_EB={theta_EB:.3e} | "
            f"TE={te_fit:.4e} | "
            f"TE_ref={te_previous_ref:.4e} | "
            f"nnz_eff={nnz_eff:4d}/{nnz_target:4d} | "
            f"sum(w)={sum_w:+.6f} | "
            f"score={score:.3f}"
        )

    return row


def run_fit_grid(
    fit_window: Dict[str, Any],
    c_grid: Sequence[float],
    te_previous_ref: float,
    nnz_target: int,
    tau_c: float = 2e-3,
    theta_bound_factor: float = 10.0,
    sapg_n_iter: int = 15_000,
    sapg_burn_pr: int = 4_000,
    sapg_c: float = 1.0,
    sapg_k0: int = 200,
    sapg_pr_q: float = 1.0,
    fista_max_iter: int = 4_000,
    fista_tol: float = 1e-6,
    nnz_eff_thresh: float = 1e-4,
    nnz_sigma: float = 25.0,
    te_hi_frac: float = 5.0,
    te_lo_frac: float = 0.0,
    power_max_iter: int = 200,
    power_tol: float = 1e-7,
    rng: Optional[np.random.Generator] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Run the full c-grid for one FIT window.

    Returns:
      out["rows"]    full rows with arrays/traces;
      out["summary"] table over c;
      out["best"]    selected row.
    """
    rng = np.random.default_rng() if rng is None else rng

    k = int(fit_window["k"])

    theta0, w_ls0 = theta0_weighted_ols_l1(
        y_c=fit_window["y_c"],
        R_c=fit_window["R_c"],
        alpha=fit_window["alpha"],
        w_ols=fit_window.get("w_ols", None),
    )

    theta_bounds = (
        theta0 / theta_bound_factor,
        theta0 * theta_bound_factor,
    )

    if verbose:
        print(
            f"\n[FIT-{k:02d} grid] "
            f"{fit_window['fit_start']} -> {fit_window['fit_end']} | "
            f"sigma2_base={float(fit_window['sigma2_ols']):.3e} | "
            f"TE_previous_ref={float(te_previous_ref):.4e} | "
            f"nnz_target={int(nnz_target)} | "
            f"theta0={theta0:.3e} | "
            f"bounds=[{theta_bounds[0]:.3e}, {theta_bounds[1]:.3e}]"
        )

    rows = []

    for c in c_grid:
        row = run_one_c(
            fit_window=fit_window,
            c=float(c),
            theta0=theta0,
            theta_bounds=theta_bounds,
            te_previous_ref=te_previous_ref,
            nnz_target=nnz_target,
            tau_c=tau_c,
            sapg_n_iter=sapg_n_iter,
            sapg_burn_pr=sapg_burn_pr,
            sapg_c=sapg_c,
            sapg_k0=sapg_k0,
            sapg_pr_q=sapg_pr_q,
            fista_max_iter=fista_max_iter,
            fista_tol=fista_tol,
            nnz_eff_thresh=nnz_eff_thresh,
            nnz_sigma=nnz_sigma,
            te_hi_frac=te_hi_frac,
            te_lo_frac=te_lo_frac,
            power_max_iter=power_max_iter,
            power_tol=power_tol,
            rng=rng,
            verbose=verbose,
        )

        rows.append(row)

    summary = make_grid_summary(rows)
    best = select_best_grid_row(rows)

    if verbose:
        warning = best.get("selection_warning", "")
        if warning:
            print(f"[FIT-{k:02d} grid][WARN] {warning}")

        print(
            f"[FIT-{k:02d} grid] selected "
            f"c*={best['c']:.3g} | "
            f"sigma2*={best['sigma2_eff']:.3e} | "
            f"theta_EB*={best['theta_EB']:.3e} | "
            f"TE={best['TE_fit']:.4e} | "
            f"TE_ref={best['TE_previous_ref']:.4e} | "
            f"nnz_eff={best['nnz_eff']} | "
            f"nnz_target={best['nnz_target']} | "
            f"score={best['score']:.3f}"
        )

    return {
        "k": k,
        "rows": rows,
        "summary": summary,
        "best": best,
        "theta0": theta0,
        "theta_bounds": theta_bounds,
        "w_ls0": w_ls0,
        "te_previous_ref": float(te_previous_ref),
        "nnz_target": int(nnz_target),
    }


def selected_summary_from_results(
    by_window: Dict[int, Dict[str, Any]],
) -> pd.DataFrame:
    """
    Build a compact table of selected c* rows across FIT windows.
    """
    records = []

    for k, out in sorted(by_window.items()):
        b = out["best"]

        records.append(
            {
                "k": k,
                "fit_start": b["fit_start"],
                "fit_end": b["fit_end"],
                "c_star": b["c"],
                "sigma2_base": b["sigma2_base"],
                "sigma2_star": b["sigma2_eff"],
                "theta0": b["theta0"],
                "theta_EB_star": b["theta_EB"],
                "L_star": b["L"],
                "lambda_MY_star": b["lambda_MY"],
                "gamma_MYULA_star": b["gamma_MYULA"],
                "TE_fit": b["TE_fit"],
                "TE_previous_ref": b["TE_previous_ref"],
                "TE_ref_OLS": b["TE_ref_OLS"],
                "TE_ratio_vs_previous": b["TE_ratio_vs_previous"],
                "TE_ratio_vs_OLS": b["TE_ratio_vs_OLS"],
                "nnz_target": b["nnz_target"],
                "nnz_raw": b["nnz_raw"],
                "nnz_eff": b["nnz_eff"],
                "weighted_l1": b["weighted_l1"],
                "sum_w": b["sum_w"],
                "budget_error": b["budget_error"],
                "score": b["score"],
                "sapg_seconds": b["sapg_seconds"],
                "fista_seconds": b["fista_seconds"],
                "fista_iters": b["fista_iters"],
            }
        )

    return pd.DataFrame(records)


def make_rolling_targets(
    fit_window: Dict[str, Any],
    previous_w: Optional[np.ndarray],
    initial_nnz_target: int = 130,
    initial_te_ref: float = 1e-5,
    nnz_eff_thresh: float = 1e-4,
) -> Tuple[float, int]:
    """
    Construct the TE and nnz targets for one FIT window.

    FIT 1:
        previous_w is None, so use:
            nnz_target = initial_nnz_target,
            TE_previous_ref = max(initial_te_ref, TE_OLS_on_this_FIT).

    FIT k >= 2:
        previous_w is the previously selected portfolio, so use:
            TE_previous_ref = TE(previous_w on current FIT window),
            nnz_target = effective nnz(previous_w).
    """
    if previous_w is None:
        te_ols = te_rmse_raw_centered(
            y_raw=fit_window["y_fit"],
            R_raw=fit_window["R_fit"],
            w=fit_window["w_ols"],
            y_mu=fit_window["y_mu"],
            R_mu=fit_window["R_mu"],
        )

        te_previous_ref = max(float(initial_te_ref), float(te_ols))
        nnz_target = int(initial_nnz_target)

        return te_previous_ref, nnz_target

    te_previous_ref = te_rmse_raw_centered(
        y_raw=fit_window["y_fit"],
        R_raw=fit_window["R_fit"],
        w=previous_w,
        y_mu=fit_window["y_mu"],
        R_mu=fit_window["R_mu"],
    )

    nnz_target = count_effective_nnz(
        previous_w,
        thresh=nnz_eff_thresh,
    )

    return float(te_previous_ref), int(nnz_target)


def run_all_fit_grids(
    fit_windows: List[Dict[str, Any]],
    c_grid: Sequence[float],
    tau_c: float = 2e-3,
    initial_nnz_target: int = 100,
    initial_te_ref: float = 1e-5,
    nnz_sigma: float = 25.0,
    nnz_eff_thresh: float = 1e-4,
    te_hi_frac: float = 5.0,
    te_lo_frac: float = 0.0,
    sapg_n_iter: int = 15_000,
    sapg_burn_pr: int = 4_000,
    fista_max_iter: int = 4_000,
    fista_tol: float = 1e-6,
    rng_seed: int = 12345,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Run variance grids over all FIT windows.

    Returns:
        out["by_window"][k]       full output for FIT window k;
        out["selected_rows"][k]   selected row for FIT window k;
        out["selected_summary"]   compact table across windows.

    Rolling convention:
        The selected MAP from FIT k is provisionally treated as the
        previously held portfolio when setting TE_previous_ref and
        nnz_target for FIT k+1.

    Later, after UQ/gating is inserted, replace

        previous_w = out_k["best"]["w_map"]

    by the gated or final held portfolio.
    """
    check_fit_windows(fit_windows)

    rng = np.random.default_rng(rng_seed)

    by_window = {}
    selected_rows = {}

    previous_w = None

    for fit_window in fit_windows:
        k = int(fit_window["k"])

        te_previous_ref_k, nnz_target_k = make_rolling_targets(
            fit_window=fit_window,
            previous_w=previous_w,
            initial_nnz_target=initial_nnz_target,
            initial_te_ref=initial_te_ref,
            nnz_eff_thresh=nnz_eff_thresh,
        )

        out_k = run_fit_grid(
            fit_window=fit_window,
            c_grid=c_grid,
            te_previous_ref=te_previous_ref_k,
            nnz_target=nnz_target_k,
            tau_c=tau_c,
            sapg_n_iter=sapg_n_iter,
            sapg_burn_pr=sapg_burn_pr,
            fista_max_iter=fista_max_iter,
            fista_tol=fista_tol,
            nnz_eff_thresh=nnz_eff_thresh,
            nnz_sigma=nnz_sigma,
            te_hi_frac=te_hi_frac,
            te_lo_frac=te_lo_frac,
            rng=rng,
            verbose=verbose,
        )

        by_window[k] = out_k
        selected_rows[k] = out_k["best"]

        previous_w = out_k["best"]["w_map"]

    selected_summary = selected_summary_from_results(by_window)

    return {
        "by_window": by_window,
        "selected_rows": selected_rows,
        "selected_summary": selected_summary,
    }

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — construction effective-variance grid
# ===============================================================

# Effective variance grid:
#   sigma2_eff(c) = c * sigma2_ols
c_grid = np.asarray(C_GRID_CONSTRUCTION, dtype=float)

# Main construction target for the pre-gating MAP frontier.
initial_nnz_target = int(NNZ_TARGET)

fit_grid_out = run_all_fit_grids(
    fit_windows=rolling_fit_with_sigma[:1],
    c_grid=c_grid,
    tau_c=TAU_C,

    # FIT-1 rolling targets.
    initial_nnz_target=initial_nnz_target,
    initial_te_ref=INITIAL_TE_REF,

    # Effective nonzero threshold and score-shape parameters.
    nnz_eff_thresh=NNZ_EFF_THRESH,
    nnz_sigma=NNZ_SIGMA,

    # Loose TE gate for the pre-gating portfolio-building stage.
    te_hi_frac=TE_HI_FRAC,
    te_lo_frac=TE_LO_FRAC,

    # Smoke vs production numerical settings.
    sapg_n_iter=CONSTRUCTION_SAPG_N_ITER,
    sapg_burn_pr=CONSTRUCTION_SAPG_BURN_PR,
    fista_max_iter=CONSTRUCTION_FISTA_MAX_ITER,
    fista_tol=CONSTRUCTION_FISTA_TOL,

    rng_seed=RANDOM_SEED,
    verbose=VERBOSE,
)

# Alias used by downstream helper cells.
fit_grid_smoke = fit_grid_out

print("\n[Selected construction summary]")
print(fit_grid_out["selected_summary"].to_string(index=False))

print("\n[Full construction grid for FIT-1]")
print(fit_grid_out["by_window"][1]["summary"].to_string(index=False))

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — Block 6
# MALA setup and helper functions
#
# This block conditions the long-run UQ sampler on the selected
# variance-grid row for one FIT window.
#
# Current workflow:
#   - use FIT 1 smoke/grid result;
#   - condition on selected c*, sigma2*, theta_EB* and w_MAP*;
#   - build a diagonal preconditioner;
#   - define the MH-corrected preconditioned MALA kernel.
#
# Important:
#   The diagonal preconditioner uses the literal Jacobi diagonal of the
#   soft-budget Hessian contribution, i.e. + 2*Lambda, not + 2*Lambda*p.
#   If MALA mixing is poor, revisit this choice.
# ===============================================================

import time
import numpy as np
import pandas as pd

from typing import Optional, Dict, Any, Iterable, Tuple, List


# ---------------------------------------------------------------
# Select the FIT window and selected grid row for UQ
# ---------------------------------------------------------------

fit_k = 1

fit_window = rolling_fit_with_sigma[fit_k - 1]
best_fit = fit_grid_smoke["selected_rows"][fit_k]

# Raw and centered FIT-window data
y_c_fit = fit_window["y_c"]
R_c_fit = fit_window["R_c"]
alpha_fit = fit_window["alpha"]

y_raw_fit = fit_window["y_fit"]
R_raw_fit = fit_window["R_fit"]
y_mu_fit = fit_window["y_mu"]
R_mu_fit = fit_window["R_mu"]

# Selected grid quantities
w_map_fit = best_fit["w_map"]
theta_EB_fit = float(best_fit["theta_EB"])
sigma2_fit = float(best_fit["sigma2_eff"])
Lambda_fit = float(best_fit["steps"]["Lambda"])
c_star_fit = float(best_fit["c"])

print(
    f"[UQ setup] FIT-{fit_k:02d} | "
    f"c*={c_star_fit:.3g} | "
    f"sigma2*={sigma2_fit:.3e} | "
    f"theta_EB*={theta_EB_fit:.3e} | "
    f"nnz_eff={best_fit['nnz_eff']} | "
    f"TE_fit={best_fit['TE_fit']:.4e}"
)


# ---------------------------------------------------------------
# Preconditioner and preconditioned Lipschitz estimate
# ---------------------------------------------------------------

def mala_build_diag_preconditioner(
    R_c: np.ndarray,
    sigma2: float,
    Lambda: float,
) -> np.ndarray:
    """
    Build a diagonal Jacobi-style preconditioner P.

    The smooth Hessian is

        A = R_c^T R_c / sigma2 + 2 Lambda 11^T.

    The literal diagonal of the budget term 2 Lambda 11^T is 2 Lambda.
    Hence we use

        D_j = ||R_{.j}||^2 / sigma2 + 2 Lambda,

    and

        P_j = 1 / sqrt(D_j).

    Note
    ----
    The older code used 2*Lambda*p as a conservative budget-direction
    scaling. We are deliberately not using that here. If mixing is poor,
    revisit this preconditioner choice.
    """
    R_c = np.asarray(R_c, dtype=float)
    diag_RtR = np.sum(R_c ** 2, axis=0)

    D = (diag_RtR / sigma2) + 2.0 * Lambda
    P = 1.0 / np.sqrt(np.clip(D, 1e-16, None))

    return P


def mala_A_unpre_matvec(
    v: np.ndarray,
    R_c: np.ndarray,
    sigma2: float,
    Lambda: float,
) -> np.ndarray:
    """
    Matrix-vector product with the unpreconditioned smooth Hessian

        A = R_c^T R_c / sigma2 + 2 Lambda 11^T.
    """
    v = np.asarray(v, dtype=float).reshape(-1)
    R_c = np.asarray(R_c, dtype=float)

    data_part = (R_c.T @ (R_c @ v)) / sigma2
    budget_part = 2.0 * Lambda * np.sum(v) * np.ones_like(v)

    return data_part + budget_part


def mala_A_precond_matvec(
    u: np.ndarray,
    R_c: np.ndarray,
    sigma2: float,
    Lambda: float,
    P: np.ndarray,
) -> np.ndarray:
    """
    Matrix-vector product with P A P in the transformed coordinates.
    """
    v = P * u
    Av = mala_A_unpre_matvec(v, R_c, sigma2, Lambda)
    return P * Av


def mala_power_iteration_precond(
    R_c: np.ndarray,
    sigma2: float,
    Lambda: float,
    P: np.ndarray,
    max_iter: int = 200,
    tol: float = 1e-7,
    rng: Optional[np.random.Generator] = None,
) -> float:
    """
    Estimate the spectral radius of P A P by power iteration.
    """
    rng = np.random.default_rng() if rng is None else rng

    p = R_c.shape[1]
    u = rng.standard_normal(p)
    u /= np.linalg.norm(u) + 1e-15

    prev = 0.0
    val = 0.0

    for _ in range(max_iter):
        Au = mala_A_precond_matvec(
            u=u,
            R_c=R_c,
            sigma2=sigma2,
            Lambda=Lambda,
            P=P,
        )

        val = float(np.dot(u, Au))
        norm_Au = np.linalg.norm(Au)

        if norm_Au <= 1e-15:
            break

        u_new = Au / norm_Au

        if abs(val - prev) <= tol * max(1.0, abs(val)):
            break

        u = u_new
        prev = val

    return float(val)


def mala_compute_precond_steps(
    R_c: np.ndarray,
    sigma2: float,
    Lambda: float,
    c_lambda: float = 1.0,
    power_max_iter: int = 200,
    power_tol: float = 1e-7,
    rng: Optional[np.random.Generator] = None,
) -> Tuple[Dict[str, float], np.ndarray]:
    """
    Compute preconditioned MALA/MY quantities.

    lambda_MY_pre is the Moreau--Yosida smoothing parameter used by the
    MALA target:

        lambda_MY_pre = c_lambda / L_pre.

    Returns
    -------
    mala_steps : dict
        Contains L_pre, lambda_MY_pre, Lambda, sigma2, etc.
    P : np.ndarray
        Diagonal preconditioner entries.
    """
    P = mala_build_diag_preconditioner(
        R_c=R_c,
        sigma2=sigma2,
        Lambda=Lambda,
    )

    L_pre = mala_power_iteration_precond(
        R_c=R_c,
        sigma2=sigma2,
        Lambda=Lambda,
        P=P,
        max_iter=power_max_iter,
        tol=power_tol,
        rng=rng,
    )

    if L_pre <= 0.0:
        raise ValueError(f"Non-positive preconditioned Lipschitz estimate: {L_pre}")

    lambda_MY_pre = float(c_lambda / L_pre)

    mala_steps = {
        "L_pre": float(L_pre),
        "lambda_MY_pre": float(lambda_MY_pre),
        "Lambda": float(Lambda),
        "sigma2": float(sigma2),
        "power_iters": int(power_max_iter),
        "power_tol": float(power_tol),
        "c_lambda": float(c_lambda),
    }

    return mala_steps, P


# ---------------------------------------------------------------
# Smoothed target, gradient, and MH-corrected MALA step
# ---------------------------------------------------------------

def mala_phi_smoothed(
    w: np.ndarray,
    y_c: np.ndarray,
    R_c: np.ndarray,
    alpha: np.ndarray,
    theta: float,
    mala_steps: Dict[str, float],
) -> float:
    """
    Smoothed negative log-posterior

        Phi_lambda(w)
        = 0.5/sigma2 ||y_c - R_c w||^2
          + Lambda (1^T w - 1)^2
          + g_lambda(w),

    where g_lambda is the Moreau--Yosida smoothing of the weighted l1 term.
    """
    sigma2 = mala_steps["sigma2"]
    Lambda = mala_steps["Lambda"]
    lambda_MY = mala_steps["lambda_MY_pre"]

    resid = y_c - R_c @ w

    f = (
        0.5 / sigma2 * float(resid @ resid)
        + Lambda * float((np.sum(w) - 1.0) ** 2)
    )

    prox = soft_threshold_weighted(
        x=w,
        lam=lambda_MY * theta,
        alpha=alpha,
    )

    g_smoothed = (
        0.5 / lambda_MY * float(np.sum((w - prox) ** 2))
        + theta * float(np.sum(alpha * np.abs(prox)))
    )

    return float(f + g_smoothed)


def mala_logpost_smoothed(
    w: np.ndarray,
    y_c: np.ndarray,
    R_c: np.ndarray,
    alpha: np.ndarray,
    theta: float,
    mala_steps: Dict[str, float],
) -> float:
    """
    Smoothed log-posterior up to an additive constant.
    """
    return -mala_phi_smoothed(
        w=w,
        y_c=y_c,
        R_c=R_c,
        alpha=alpha,
        theta=theta,
        mala_steps=mala_steps,
    )


def mala_grad_phi_smoothed(
    w: np.ndarray,
    y_c: np.ndarray,
    R_c: np.ndarray,
    alpha: np.ndarray,
    theta: float,
    mala_steps: Dict[str, float],
) -> np.ndarray:
    """
    Gradient of the smoothed negative log-posterior.
    """
    sigma2 = mala_steps["sigma2"]
    Lambda = mala_steps["Lambda"]
    lambda_MY = mala_steps["lambda_MY_pre"]

    grad_f = grad_smooth_budget(
        y_c=y_c,
        R_c=R_c,
        w=w,
        sigma2=sigma2,
        Lambda=Lambda,
    )

    prox = soft_threshold_weighted(
        x=w,
        lam=lambda_MY * theta,
        alpha=alpha,
    )

    grad_g_smoothed = (w - prox) / lambda_MY

    return grad_f + grad_g_smoothed


def mala_mh_step(
    w: np.ndarray,
    y_c: np.ndarray,
    R_c: np.ndarray,
    alpha: np.ndarray,
    theta: float,
    mala_steps: Dict[str, float],
    P: np.ndarray,
    gamma: float,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, bool, float]:
    """
    One MH-corrected preconditioned MALA step.

    Proposal:

        w' = w - gamma P^2 grad Phi(w)
             + sqrt(2 gamma) P xi,

    where xi ~ N(0, I).
    """
    P = np.asarray(P, dtype=float)
    P_safe = np.clip(P, 1e-16, None)
    P2 = P_safe * P_safe

    grad_w = mala_grad_phi_smoothed(
        w=w,
        y_c=y_c,
        R_c=R_c,
        alpha=alpha,
        theta=theta,
        mala_steps=mala_steps,
    )

    mean_fwd = w - gamma * (P2 * grad_w)
    cand = mean_fwd + np.sqrt(2.0 * gamma) * (P_safe * rng.normal(size=w.shape))

    Phi_w = mala_phi_smoothed(
        w=w,
        y_c=y_c,
        R_c=R_c,
        alpha=alpha,
        theta=theta,
        mala_steps=mala_steps,
    )

    Phi_c = mala_phi_smoothed(
        w=cand,
        y_c=y_c,
        R_c=R_c,
        alpha=alpha,
        theta=theta,
        mala_steps=mala_steps,
    )

    grad_c = mala_grad_phi_smoothed(
        w=cand,
        y_c=y_c,
        R_c=R_c,
        alpha=alpha,
        theta=theta,
        mala_steps=mala_steps,
    )

    mean_rev = cand - gamma * (P2 * grad_c)

    def proposal_quad(x: np.ndarray, mean: np.ndarray) -> float:
        z = (x - mean) / (np.sqrt(2.0 * gamma) * P_safe)
        return 0.5 * float(z @ z)

    log_q_fwd = -proposal_quad(cand, mean_fwd)
    log_q_rev = -proposal_quad(w, mean_rev)

    log_alpha = -(Phi_c - Phi_w) + (log_q_rev - log_q_fwd)

    accept = bool(np.log(rng.random()) < log_alpha)

    if accept:
        return cand, True, float(log_alpha)

    return w, False, float(log_alpha)


# ---------------------------------------------------------------
# Burn-in and simple time-series diagnostics
# ---------------------------------------------------------------

def mala_burnin(
    y_c: np.ndarray,
    R_c: np.ndarray,
    alpha: np.ndarray,
    theta: float,
    mala_steps: Dict[str, float],
    P: np.ndarray,
    gamma: float,
    n_burn: int = 20_000,
    rng: Optional[np.random.Generator] = None,
    w_init: Optional[np.ndarray] = None,
) -> np.ndarray:
    """
    Plain MALA burn-in for the smoothed target.

    Runs n_burn MALA steps starting from w_init and returns the final state.
    No samples are stored.
    """
    rng = np.random.default_rng() if rng is None else rng

    p = R_c.shape[1]

    if w_init is None:
        w = np.zeros(p, dtype=float)
    else:
        w = np.asarray(w_init, dtype=float).reshape(-1).copy()

    for _ in range(n_burn):
        w, _, _ = mala_mh_step(
            w=w,
            y_c=y_c,
            R_c=R_c,
            alpha=alpha,
            theta=theta,
            mala_steps=mala_steps,
            P=P,
            gamma=gamma,
            rng=rng,
        )

    return w


def acf_1d(
    x: np.ndarray,
    max_lag: int = 400,
) -> np.ndarray:
    """
    Simple autocorrelation function for one scalar series.
    """
    x = np.asarray(x, dtype=float).reshape(-1)
    n = len(x)

    if n == 0:
        return np.zeros(max_lag + 1)

    max_lag = int(min(max_lag, max(n - 1, 0)))

    x0 = x - np.mean(x)
    var = float((x0 @ x0) / max(n, 1))

    if var <= 1e-20:
        return np.zeros(max_lag + 1)

    r = np.empty(max_lag + 1)

    for lag in range(max_lag + 1):
        r[lag] = float((x0[: n - lag] @ x0[lag:]) / (n * var + 1e-15))

    return r


def ess_1d(
    x: np.ndarray,
    max_lag: int = 400,
) -> float:
    """
    Initial-positive-sequence-style ESS estimate for one scalar series.
    """
    x = np.asarray(x, dtype=float).reshape(-1)

    if len(x) <= 1:
        return float(len(x))

    r = acf_1d(x, max_lag=max_lag)

    s = 0.0

    for lag in range(1, len(r)):
        if r[lag] < 0:
            break
        s += r[lag]

    tau = 1.0 + 2.0 * s

    return float(len(x) / max(tau, 1e-12))


def te_series_from_samples_centered(
    y_c: np.ndarray,
    R_c: np.ndarray,
    W: np.ndarray,
) -> np.ndarray:
    """
    TE series for samples W using centered data.

    This equals the raw-return TE under the stored centering convention:

        y_raw - [y_mu + (R_raw - R_mu)w] = y_c - R_c w.
    """
    y_c = np.asarray(y_c, dtype=float).reshape(-1)
    R_c = np.asarray(R_c, dtype=float)
    W = np.asarray(W, dtype=float)

    preds = R_c @ W.T
    resid = y_c[:, None] - preds

    return np.sqrt(np.mean(resid ** 2, axis=0)).reshape(-1)


# ---------------------------------------------------------------
# Build MALA preconditioner for the selected FIT row
# ---------------------------------------------------------------

rng_mala_setup = np.random.default_rng(2026)

mala_steps, mala_P = mala_compute_precond_steps(
    R_c=R_c_fit,
    sigma2=sigma2_fit,
    Lambda=Lambda_fit,
    c_lambda=1.0,
    power_max_iter=200,
    power_tol=1e-7,
    rng=rng_mala_setup,
)

print(
    f"[MALA setup] L_pre={mala_steps['L_pre']:.3e} | "
    f"lambda_MY_pre={mala_steps['lambda_MY_pre']:.3e} | "
    f"sigma2={mala_steps['sigma2']:.3e} | "
    f"Lambda={mala_steps['Lambda']:.3e}"
)

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — Block 7
# Gamma tuning for preconditioned MALA
#
# We run short fixed-length MALA chains for a grid of gamma multipliers
# and select a gamma whose acceptance rate is close to the target range.
#
# If all acceptance rates are too high, the grid is automatically expanded.
# ===============================================================


def mala_default_gamma(
    mala_steps: Dict[str, float],
    safety: float = 0.9,
) -> float:
    """
    Conservative base gamma based on the preconditioned data curvature
    and the MY smoothing curvature.

    If lambda_MY_pre = 1 / L_pre, this is approximately safety / (2 L_pre).
    """
    L_pre = float(mala_steps["L_pre"])
    lambda_MY_pre = float(mala_steps["lambda_MY_pre"])

    L_total = L_pre + 1.0 / lambda_MY_pre

    return float(safety / L_total)


def mala_short_run_acceptance(
    y_c: np.ndarray,
    R_c: np.ndarray,
    alpha: np.ndarray,
    theta: float,
    mala_steps: Dict[str, float],
    P: np.ndarray,
    gamma: float,
    n_steps: int = 2_000,
    rng: Optional[np.random.Generator] = None,
    w_init: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    """
    Run a short MALA chain and return acceptance diagnostics.

    Each tuning candidate starts from w_init. This keeps the comparison
    between candidate gammas simple and reproducible.
    """
    rng = np.random.default_rng() if rng is None else rng

    p = R_c.shape[1]

    if w_init is None:
        w = np.zeros(p, dtype=float)
    else:
        w = np.asarray(w_init, dtype=float).reshape(-1).copy()

    acc = 0
    log_alpha_vals = np.empty(n_steps)

    for it in range(n_steps):
        w, accepted, log_alpha = mala_mh_step(
            w=w,
            y_c=y_c,
            R_c=R_c,
            alpha=alpha,
            theta=theta,
            mala_steps=mala_steps,
            P=P,
            gamma=gamma,
            rng=rng,
        )

        acc += int(accepted)
        log_alpha_vals[it] = log_alpha

    acc_rate = acc / max(n_steps, 1)

    return {
        "gamma": float(gamma),
        "accept_rate": float(acc_rate),
        "mean_log_alpha": float(np.mean(log_alpha_vals)),
        "median_log_alpha": float(np.median(log_alpha_vals)),
        "final_w": w,
    }


def mala_tune_gamma(
    y_c: np.ndarray,
    R_c: np.ndarray,
    alpha: np.ndarray,
    theta: float,
    mala_steps: Dict[str, float],
    P: np.ndarray,
    w_init: np.ndarray,
    gamma_base: Optional[float] = None,
    gamma_factors: Optional[Iterable[float]] = None,
    n_steps: int = 2_000,
    target_accept: float = 0.60,
    accept_window: Tuple[float, float] = (0.55, 0.65),
    max_expansions: int = 6,
    rng_seed: int = 2027,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Tune gamma by short fixed-length MALA runs.

    Selection rule:
      - prefer candidates inside accept_window;
      - among those, choose closest to target_accept;
      - if none are inside the window, choose closest to target_accept.

    Automatic expansion:
      - if all tested acceptance rates are above accept_window[1],
        the largest gamma factor is doubled and another candidate is tested;
      - this continues up to max_expansions.

    This is useful when the initial gamma grid is too conservative.
    """
    if gamma_base is None:
        gamma_base = mala_default_gamma(mala_steps)

    if gamma_factors is None:
        gamma_factors = [1.0, 2.0, 4.0, 8.0, 16.0, 32.0]

    gamma_factors = list(gamma_factors)

    rows = []
    tested_factors = set()

    rng_master = np.random.default_rng(rng_seed)

    if verbose:
        print(
            f"[MALA gamma tune] gamma_base={gamma_base:.3e} | "
            f"target_accept={target_accept:.2f} | "
            f"accept_window=({accept_window[0]:.2f}, {accept_window[1]:.2f})"
        )

    def run_candidate(factor: float) -> Dict[str, float]:
        gamma = float(factor) * float(gamma_base)

        rng_candidate = np.random.default_rng(
            rng_master.integers(0, 2**32 - 1)
        )

        out = mala_short_run_acceptance(
            y_c=y_c,
            R_c=R_c,
            alpha=alpha,
            theta=theta,
            mala_steps=mala_steps,
            P=P,
            gamma=gamma,
            n_steps=n_steps,
            rng=rng_candidate,
            w_init=w_init,
        )

        row = {
            "factor": float(factor),
            "gamma": float(gamma),
            "accept_rate": float(out["accept_rate"]),
            "mean_log_alpha": float(out["mean_log_alpha"]),
            "median_log_alpha": float(out["median_log_alpha"]),
        }

        if verbose:
            print(
                f"[MALA gamma tune] "
                f"factor={factor:8.2f} | "
                f"gamma={gamma:.3e} | "
                f"accept={out['accept_rate']:.3f} | "
                f"mean log-alpha={out['mean_log_alpha']:.3f}"
            )

        return row

    # Run initial grid.
    for factor in gamma_factors:
        factor = float(factor)

        if factor in tested_factors:
            continue

        rows.append(run_candidate(factor))
        tested_factors.add(factor)

    lo, hi = accept_window

    # Expand if everything is still too conservative.
    expansions = 0

    while (
        all(row["accept_rate"] > hi for row in rows)
        and expansions < max_expansions
    ):
        next_factor = 2.0 * max(row["factor"] for row in rows)

        if verbose:
            print(
                f"[MALA gamma tune] all acceptances > {hi:.2f}; "
                f"expanding to factor={next_factor:.2f}"
            )

        rows.append(run_candidate(next_factor))
        tested_factors.add(next_factor)
        expansions += 1

    tune_table = pd.DataFrame(rows).sort_values("factor").reset_index(drop=True)

    inside = tune_table[
        (tune_table["accept_rate"] >= lo)
        & (tune_table["accept_rate"] <= hi)
    ].copy()

    if len(inside) > 0:
        idx = (inside["accept_rate"] - target_accept).abs().idxmin()
        selection_note = "selected from candidates inside acceptance window"
    else:
        idx = (tune_table["accept_rate"] - target_accept).abs().idxmin()
        selection_note = "no candidate inside acceptance window; selected closest to target"

    selected = tune_table.loc[idx].to_dict()

    if verbose:
        print(
            f"[MALA gamma tune] selected gamma={selected['gamma']:.3e} | "
            f"factor={selected['factor']:.2f} | "
            f"accept={selected['accept_rate']:.3f} | "
            f"{selection_note}"
        )

        if selected["accept_rate"] > hi:
            print(
                "[MALA gamma tune][NOTE] selected acceptance is still high. "
                "If ESS/mixing is poor, revisit the preconditioner and/or "
                "increase max_expansions."
            )

    return {
        "table": tune_table,
        "selected": selected,
        "gamma": float(selected["gamma"]),
        "gamma_base": float(gamma_base),
        "selection_note": selection_note,
    }


# ---------------------------------------------------------------
# Run gamma tuning
# ---------------------------------------------------------------

gamma_tune = mala_tune_gamma(
    y_c=y_c_fit,
    R_c=R_c_fit,
    alpha=alpha_fit,
    theta=theta_EB_fit,
    mala_steps=mala_steps,
    P=mala_P,
    w_init=w_map_fit,

    # Let the function compute the conservative base gamma.
    gamma_base=None,

    # Wider starting grid than before.
    gamma_factors=[1.0, 2.0, 4.0, 8.0, 16.0, 32.0],

    # Short tuning chains.
    n_steps=MALA_TUNE_STEPS,

    # Target for MALA acceptance.
    target_accept=0.60,
    accept_window=(0.55, 0.65),

    # Allow automatic expansion if all acceptances remain too high.
    max_expansions=6,

    rng_seed=2027,
    verbose=True,
)

mala_gamma = gamma_tune["gamma"]

print(f"[MALA gamma] selected gamma = {mala_gamma:.3e}")
print(gamma_tune["table"].to_string(index=False))

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — Block 8
# Long MALA run with ESS stopping rule
#
# Augmented run:
#   n_iter increased from 250_000 to 450_000.
#
# Stopping targets:
#   - min ESS over monitored coordinates >= ess_min_S
#   - ESS(TE) >= ess_min_TE
# ===============================================================


def mala_run_with_stopping(
    y_c: np.ndarray,
    R_c: np.ndarray,
    alpha: np.ndarray,
    theta: float,
    mala_steps: Dict[str, float],
    P: np.ndarray,
    gamma: float,
    w_init: np.ndarray,
    s_indices: Optional[Iterable[int]] = None,
    thin: int = 1,
    n_iter: int = 450_000,
    check_every_kept: int = 2_000,
    ess_min_S: float = 400.0,
    ess_min_TE: float = 1_000.0,
    max_lag: int = 200,
    n_burn: int = 20_000,
    rng: Optional[np.random.Generator] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Long preconditioned MALA run with ESS-based stopping.

    Parameters
    ----------
    w_init : np.ndarray
        Starting point. For the current workflow this is the selected MAP
        from the variance-grid stage.
    s_indices : iterable of int, optional
        Coordinates monitored for coordinate-wise ESS. If None, use the
        top 200 absolute entries of w_init.
    """
    rng = np.random.default_rng() if rng is None else rng

    y_c = np.asarray(y_c, dtype=float).reshape(-1)
    R_c = np.asarray(R_c, dtype=float)
    alpha = np.asarray(alpha, dtype=float).reshape(-1)
    w_init = np.asarray(w_init, dtype=float).reshape(-1)

    p = R_c.shape[1]
    T = R_c.shape[0]

    if s_indices is None:
        s_indices = np.argsort(-np.abs(w_init))[: min(200, p)]

    s_indices = np.array(list(s_indices), dtype=int)

    if verbose:
        print(
            f"[MALA long] p={p}, monitored coords={len(s_indices)}, "
            f"n_burn={n_burn}, n_iter={n_iter}, thin={thin}"
        )

    # Burn-in from the selected MAP.
    if n_burn > 0:
        if verbose:
            print("[MALA long] starting burn-in...")

        w = mala_burnin(
            y_c=y_c,
            R_c=R_c,
            alpha=alpha,
            theta=theta,
            mala_steps=mala_steps,
            P=P,
            gamma=gamma,
            n_burn=n_burn,
            rng=rng,
            w_init=w_init,
        )

        if verbose:
            print("[MALA long] burn-in complete.")
    else:
        w = w_init.copy()

    n_keep = n_iter // thin

    W = np.empty((n_keep, p), dtype=float)
    S_trace = np.empty(n_keep, dtype=float)
    F_trace = np.empty(n_keep, dtype=float)
    LP_trace = np.empty(n_keep, dtype=float)
    TE_trace = np.empty(n_keep, dtype=float)
    sum_w_trace = np.empty(n_keep, dtype=float)

    j = 0
    acc = 0
    actual_iters = 0
    stopped = False

    t0 = time.perf_counter()

    for it in range(1, n_iter + 1):
        w, accepted, _ = mala_mh_step(
            w=w,
            y_c=y_c,
            R_c=R_c,
            alpha=alpha,
            theta=theta,
            mala_steps=mala_steps,
            P=P,
            gamma=gamma,
            rng=rng,
        )

        actual_iters = it
        acc += int(accepted)

        if (it % thin) == 0:
            resid = y_c - R_c @ w
            rss = float(resid @ resid)
            budget_error = float(np.sum(w) - 1.0)

            W[j] = w
            S_trace[j] = float(np.sum(alpha * np.abs(w)))
            F_trace[j] = (
                0.5 / mala_steps["sigma2"] * rss
                + mala_steps["Lambda"] * budget_error ** 2
            )
            LP_trace[j] = mala_logpost_smoothed(
                w=w,
                y_c=y_c,
                R_c=R_c,
                alpha=alpha,
                theta=theta,
                mala_steps=mala_steps,
            )
            TE_trace[j] = float(np.sqrt(rss / T))
            sum_w_trace[j] = float(np.sum(w))

            j += 1

            if (j % check_every_kept) == 0:
                ess_te = ess_1d(TE_trace[:j], max_lag=max_lag)
                ess_coords = [
                    ess_1d(W[:j, idx], max_lag=max_lag)
                    for idx in s_indices
                ]
                ess_min = float(np.min(ess_coords))

                if verbose:
                    elapsed = time.perf_counter() - t0
                    acc_rate = acc / max(actual_iters, 1)

                    print(
                        f"[kept {j:>7}/{n_keep}] "
                        f"ESS_min(S)={ess_min:7.1f} | "
                        f"ESS(TE)={ess_te:7.1f} | "
                        f"acc={acc_rate:5.3f} | "
                        f"gamma={gamma:.3e} | "
                        f"lambda_MY={mala_steps['lambda_MY_pre']:.3e} | "
                        f"elapsed={elapsed:.1f}s"
                    )

                if (ess_min >= ess_min_S) and (ess_te >= ess_min_TE):
                    stopped = True

                    if verbose:
                        print("[MALA long] stopping rule satisfied.")

                    break

    wall = time.perf_counter() - t0

    W = W[:j]
    S_trace = S_trace[:j]
    F_trace = F_trace[:j]
    LP_trace = LP_trace[:j]
    TE_trace = TE_trace[:j]
    sum_w_trace = sum_w_trace[:j]

    acc_overall = acc / max(actual_iters, 1)

    return {
        "W": W,
        "S": S_trace,
        "F": F_trace,
        "LP": LP_trace,
        "TE": TE_trace,
        "sum_w": sum_w_trace,
        "s_indices": s_indices,
        "wall_seconds": float(wall),
        "acc_overall": float(acc_overall),
        "actual_iters": int(actual_iters),
        "kept": int(j),
        "stopped": bool(stopped),
        "gamma": float(gamma),
        "mala_steps": dict(mala_steps),
    }


# ---------------------------------------------------------------
# Run augmented long MALA chain
# ---------------------------------------------------------------

s_monitor = np.argsort(-np.abs(w_map_fit))[: min(200, len(w_map_fit))]

mala_out = mala_run_with_stopping(
    y_c=y_c_fit,
    R_c=R_c_fit,
    alpha=alpha_fit,
    theta=theta_EB_fit,
    mala_steps=mala_steps,
    P=mala_P,
    gamma=mala_gamma,
    w_init=w_map_fit,
    s_indices=s_monitor,
    thin=CONSTRUCTION_MALA_THIN,
    n_iter=CONSTRUCTION_MALA_N_ITER,
    check_every_kept=2_000,
    ess_min_S=CONSTRUCTION_ESS_MIN_S,
    ess_min_TE=CONSTRUCTION_ESS_MIN_TE,
    max_lag=CONSTRUCTION_MAX_LAG_ESS,
    n_burn=CONSTRUCTION_MALA_N_BURN,
    rng=np.random.default_rng(2028),
    verbose=True,
)

print(
    f"[MALA long] wall={mala_out['wall_seconds']:.1f}s | "
    f"kept={mala_out['kept']} | "
    f"actual_iters={mala_out['actual_iters']} | "
    f"acc={mala_out['acc_overall']:.3f} | "
    f"stopped={mala_out['stopped']}"
)

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — Block 9
# ESS and MCSE diagnostics for MALA output
# ===============================================================


def mala_report_ess_summary(
    mala_out: Dict[str, Any],
    coord_idxs: Optional[Iterable[int]] = None,
    max_lag: int = 200,
    wall_seconds: Optional[float] = None,
) -> Dict[str, Any]:
    """
    Report coordinate ESS and TE ESS for a MALA output dictionary.
    """
    W = mala_out["W"]
    TE = mala_out["TE"]

    n_keep, p = W.shape

    if coord_idxs is None:
        coord_idxs = np.linspace(0, p - 1, min(10, p), dtype=int)

    coord_idxs = np.array(list(coord_idxs), dtype=int)

    ess_coords = [
        ess_1d(W[:, j], max_lag=max_lag)
        for j in coord_idxs
    ]

    ess_te = ess_1d(TE, max_lag=max_lag)

    print(f"Kept samples: {n_keep}")
    print(f"Acceptance rate: {mala_out['acc_overall']:.3f}")
    print(f"ESS(coords {list(coord_idxs)}): {[f'{e:.1f}' for e in ess_coords]}")
    print(f"ESS(TE): {ess_te:.1f}")

    if wall_seconds is None:
        wall_seconds = mala_out.get("wall_seconds", None)

    if wall_seconds is not None and wall_seconds > 0:
        print(f"ESS/sec (TE): {ess_te / wall_seconds:.3f}")

    return {
        "coord_idxs": coord_idxs,
        "ess_coords": ess_coords,
        "ess_te": float(ess_te),
    }


def mala_mcse_te(
    mala_out: Dict[str, Any],
    max_lag: int = 200,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Compute MCSE for the posterior mean of TE.
    """
    te_series = np.asarray(mala_out["TE"], dtype=float).reshape(-1)

    sd_te = float(np.std(te_series, ddof=1))
    ess_te = float(ess_1d(te_series, max_lag=max_lag))
    mcse_te = float(sd_te / np.sqrt(max(ess_te, 1e-12)))

    if verbose:
        print(f"TE mean   = {np.mean(te_series):.4e}")
        print(f"TE sd     = {sd_te:.4e}")
        print(f"ESS(TE)   = {ess_te:.1f}")
        print(f"MCSE(TE)  = {mcse_te:.4e}")

    return {
        "te_series": te_series,
        "mean_te": float(np.mean(te_series)),
        "sd_te": sd_te,
        "ess_te": ess_te,
        "mcse_te": mcse_te,
    }


# ---------------------------------------------------------------
# Run diagnostics
# ---------------------------------------------------------------

ess_summary = mala_report_ess_summary(
    mala_out=mala_out,
    coord_idxs=mala_out["s_indices"][:10],
    max_lag=200,
    wall_seconds=mala_out["wall_seconds"],
)

mcse_te = mala_mcse_te(
    mala_out=mala_out,
    max_lag=200,
    verbose=True,
)

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — Block 9A
# Plots for long MALA run and selected gated coordinates
# ===============================================================

import os
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt


FIGDIR_MALA_FIT = Path("/content/fig_MALA_FIT1")
FIGDIR_MALA_FIT.mkdir(parents=True, exist_ok=True)


def plot_series_trace(
    series: np.ndarray,
    *,
    xlabel: str,
    ylabel: str,
    title: str,
    fname: str,
    figdir: Path = FIGDIR_MALA_FIT,
    lw: float = 0.8,
) -> None:
    """
    Generic trace plot.
    """
    series = np.asarray(series, dtype=float).reshape(-1)

    plt.figure(figsize=(6, 3.2))
    plt.plot(series, lw=lw)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(figdir / fname, dpi=160)
    plt.close()


def acf_1d_for_plot(
    x: np.ndarray,
    max_lag: int = 200,
) -> np.ndarray:
    """
    ACF helper for plotting.
    """
    x = np.asarray(x, dtype=float).reshape(-1)
    x = x - np.mean(x)

    n = len(x)
    max_lag = min(max_lag, n - 1)

    var = float((x @ x) / max(n, 1))

    if var <= 1e-20:
        return np.zeros(max_lag + 1)

    r = np.empty(max_lag + 1)

    for lag in range(max_lag + 1):
        r[lag] = float((x[: n - lag] @ x[lag:]) / (n * var + 1e-15))

    return r


def plot_acf_series(
    x: np.ndarray,
    *,
    max_lag: int = 200,
    title: str,
    fname: str,
    figdir: Path = FIGDIR_MALA_FIT,
) -> None:
    """
    ACF plot.
    """
    acf_vals = acf_1d_for_plot(x, max_lag=max_lag)

    plt.figure(figsize=(6, 3.2))
    plt.stem(np.arange(len(acf_vals)), acf_vals, basefmt=" ")
    plt.xlabel("lag")
    plt.ylabel("ACF")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(figdir / fname, dpi=160)
    plt.close()


def plot_mala_coord_trace(
    W: np.ndarray,
    j: int,
    *,
    fname: str,
    figdir: Path = FIGDIR_MALA_FIT,
) -> None:
    """
    Coordinate trace plot for one weight.
    """
    plt.figure(figsize=(6, 3.2))
    plt.plot(W[:, j], lw=0.7)
    plt.xlabel("kept draw")
    plt.ylabel(fr"$w_{{{j}}}$")
    plt.title(fr"MALA trace of $w_{{{j}}}$")
    plt.tight_layout()
    plt.savefig(figdir / fname, dpi=160)
    plt.close()


# ---------------------------------------------------------------
# Main MALA traces
# ---------------------------------------------------------------

plot_series_trace(
    mala_out["LP"],
    xlabel="kept draw",
    ylabel="log posterior",
    title="MALA long run: log posterior",
    fname="mala_logpost_longrun.png",
)

plot_series_trace(
    mala_out["S"],
    xlabel="kept draw",
    ylabel=r"$\sum_j \alpha_j |w_j|$",
    title="MALA long run: weighted-L1 trace",
    fname="mala_weighted_l1_trace.png",
)

plot_series_trace(
    mala_out["TE"],
    xlabel="kept draw",
    ylabel="TE",
    title="MALA long run: TE path on FIT",
    fname="mala_TE_trace.png",
)

plot_acf_series(
    mala_out["TE"],
    max_lag=200,
    title="ACF of TE",
    fname="acf_TE_fit.png",
)


# ---------------------------------------------------------------
# Zip figure folder
# ---------------------------------------------------------------

folder_path = str(FIGDIR_MALA_FIT)
zip_path = "./fig_MALA_FIT1.zip"

if os.path.exists(folder_path):
    shutil.make_archive(zip_path.replace(".zip", ""), "zip", folder_path)
    print(f"Folder '{folder_path}' has been zipped to '{zip_path}'")
else:
    print(f"Folder '{folder_path}' not found.")

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — Block 10b
# Practical-positive support candidates for post-MALA MAP inspection
#
# Purpose:
#   Build posterior-informed support candidates using
#
#       S(eps, pi_pos) = {j : w_MAP,j > eps and P(w_j > eps | y) >= pi_pos}.
#
#   This block:
#     - does NOT refit;
#     - does NOT lock a final portfolio;
#     - prepares selected_support_fit for the next refit block;
#     - includes ticker names where available;
#     - prints posterior confidence P(w_j > eps | y) to 3 decimals;
#     - explicitly lists:
#         (i) selected names outside the top-50 MAP;
#        (ii) top-50 MAP names not selected.
#
# Inputs assumed:
#   - mala_out
#   - w_map_fit
#
# Optional ticker sources:
#   - asset_names / tickers / asset_cols / asset_columns
#   - asset_names_fit / tickers_fit
#   - df_ret + idx_name, with assets first and index column last
# ===============================================================

from typing import Dict, Any, List
from pathlib import Path

import numpy as np
import pandas as pd


# ---------------------------------------------------------------
# 0) Basic objects
# ---------------------------------------------------------------

W_long = np.asarray(mala_out["W"], dtype=float)
w_map = np.asarray(w_map_fit, dtype=float).reshape(-1)

fit_k_current = int(globals().get("fit_k", 1))

if W_long.ndim != 2:
    raise ValueError("mala_out['W'] must be a 2D array with shape (n_samples, p).")

if W_long.shape[1] != w_map.size:
    raise ValueError(
        f"W_long has {W_long.shape[1]} columns, "
        f"but w_map_fit has length {w_map.size}."
    )

n_samples, p = W_long.shape
abs_w_map = np.abs(w_map)
L1_map = float(np.sum(abs_w_map) + 1e-16)


# ---------------------------------------------------------------
# 1) Recover ticker names if available
# ---------------------------------------------------------------

def get_optional_tickers_fit(p: int) -> List[str]:
    """
    Try common variable names used in the notebooks.

    Falls back to df_ret convention:
      - asset columns first;
      - index column named idx_name or last column.

    If no names are found, returns asset_0, asset_1, ...
    """
    candidate_names = [
        "asset_names",
        "tickers",
        "asset_cols",
        "asset_columns",
        "asset_names_fit",
        "tickers_fit",
    ]

    for name in candidate_names:
        if name in globals():
            val = list(globals()[name])
            if len(val) == p:
                return [str(x) for x in val]

    if "df_ret" in globals():
        cols = list(df_ret.columns)

        if "idx_name" in globals() and idx_name in cols:
            asset_cols_from_df = [c for c in cols if c != idx_name]
        else:
            asset_cols_from_df = cols[:-1]

        if len(asset_cols_from_df) == p:
            return [str(x) for x in asset_cols_from_df]

    return [f"asset_{j}" for j in range(p)]


tickers_fit = get_optional_tickers_fit(p)


# ---------------------------------------------------------------
# 2) Posterior probabilities for practical positivity
# ---------------------------------------------------------------

eps_list = sorted(set([float(PRACTICAL_POS_EPS), 1e-3, 2.5e-3, 5e-3, 1e-2]))
pi_pos_list = sorted(set([float(PRACTICAL_POS_PROB), 0.60, 0.65, 0.70, 0.80]))

P_pos_eps = {
    eps: (W_long > eps).mean(axis=0)
    for eps in eps_list
}


def map_l1_mass_share(S_idx: np.ndarray) -> float:
    """
    Fraction of MAP l1 mass retained by S_idx.
    """
    S_idx = np.asarray(S_idx, dtype=int)

    if S_idx.size == 0:
        return 0.0

    return float(np.sum(abs_w_map[S_idx]) / L1_map)


def make_rule_name(eps: float, pi_pos: float) -> str:
    """
    Stable candidate-rule name.
    """
    if np.isclose(eps, 1e-3):
        eps_tag = "1e-3"
    elif np.isclose(eps, 2.5e-3):
        eps_tag = "2p5e-3"
    elif np.isclose(eps, 5e-3):
        eps_tag = "5e-3"
    elif np.isclose(eps, 1e-2):
        eps_tag = "1e-2"
    else:
        eps_tag = f"{eps:.0e}".replace("-0", "-")

    return f"pos_eps{eps_tag}_pi{pi_pos:.2f}"


# ---------------------------------------------------------------
# 3) Build compact practical-positive frontier
# ---------------------------------------------------------------

records_frontier = []
candidate_rules = {}

top50_map = np.argsort(-abs_w_map)[:50]

for eps in eps_list:
    P_pos = P_pos_eps[eps]

    for pi_pos in pi_pos_list:
        keep = (w_map > eps) & (P_pos >= pi_pos)
        S = np.where(keep)[0]

        rule_name = make_rule_name(eps, pi_pos)
        candidate_rules[rule_name] = S

        records_frontier.append(
            {
                "eps": float(eps),
                "pi_pos": float(pi_pos),
                "S": int(S.size),
                "mass_kept": map_l1_mass_share(S),
                "mean_P_pos": float(np.mean(P_pos[S])) if S.size else np.nan,
                "min_P_pos": float(np.min(P_pos[S])) if S.size else np.nan,
                "n_from_top50_MAP": int(np.sum(np.isin(S, top50_map))),
            }
        )

df_practical_frontier = pd.DataFrame(records_frontier)

df_practical_frontier_display = df_practical_frontier.copy()
df_practical_frontier_display["eps"] = df_practical_frontier_display["eps"].map(
    lambda x: f"{x:.1e}"
)
df_practical_frontier_display["pi_pos"] = df_practical_frontier_display["pi_pos"].map(
    lambda x: f"{x:.2f}"
)
for col in ["mass_kept", "mean_P_pos", "min_P_pos"]:
    df_practical_frontier_display[col] = df_practical_frontier_display[col].map(
        lambda x: "" if pd.isna(x) else f"{float(x):.3f}"
    )

print("\n=== Compact practical-positive frontier ===")
print(df_practical_frontier_display.to_string(index=False))

print("\n=== |S| by eps and pi_pos ===")
print(
    df_practical_frontier
    .pivot(index="eps", columns="pi_pos", values="S")
    .to_string()
)

print("\n=== MAP l1 mass retained by eps and pi_pos ===")
print(
    df_practical_frontier
    .pivot(index="eps", columns="pi_pos", values="mass_kept")
    .to_string(float_format=lambda x: f"{x:.3f}")
)


# ---------------------------------------------------------------
# 4) Named candidate supports for next refit block
# ---------------------------------------------------------------

selected_rule_alias_fit = (
    f"selected_eps{PRACTICAL_POS_EPS:.0e}_pi{PRACTICAL_POS_PROB:.2f}"
    .replace("-0", "-")
    .replace(".", "p")
)

candidate_aliases = {
    selected_rule_alias_fit: make_rule_name(PRACTICAL_POS_EPS, PRACTICAL_POS_PROB),
    "broad_eps1e-3_pi0.60": make_rule_name(1e-3, 0.60),
    "middle_eps2p5e-3_pi0.60": make_rule_name(2.5e-3, 0.60),
    "conservative_eps5e-3_pi0.60": make_rule_name(5e-3, 0.60),
}

candidate_alias_params = {
    selected_rule_alias_fit: (float(PRACTICAL_POS_EPS), float(PRACTICAL_POS_PROB)),
    "broad_eps1e-3_pi0.60": (1e-3, 0.60),
    "middle_eps2p5e-3_pi0.60": (2.5e-3, 0.60),
    "conservative_eps5e-3_pi0.60": (5e-3, 0.60),
}

practical_support_candidates_fit = {}

for alias, rule_name in candidate_aliases.items():
    eps, pi_pos = candidate_alias_params[alias]
    S = np.asarray(candidate_rules[rule_name], dtype=int)
    P_pos = P_pos_eps[eps]

    practical_support_candidates_fit[alias] = {
        "S_idx": S,
        "eps": float(eps),
        "pi_pos": float(pi_pos),
        "rule_name": rule_name,
        "mass_kept": map_l1_mass_share(S),
        "P_pos": P_pos,
    }

print("\n=== Candidate supports prepared for next refit block ===")
for alias, obj in practical_support_candidates_fit.items():
    S = obj["S_idx"]
    print(
        f"{alias:34s} | "
        f"|S|={len(S):3d} | "
        f"MAP mass kept={obj['mass_kept']:.3f}"
    )


# ---------------------------------------------------------------
# 5) Choose one candidate for the next block
# ---------------------------------------------------------------

candidate_name_fit = selected_rule_alias_fit

# Alternatives kept for later:
# candidate_name_fit = "middle_eps2p5e-3_pi0.60"
# candidate_name_fit = "conservative_eps5e-3_pi0.60"

selected_obj = practical_support_candidates_fit[candidate_name_fit]
S_selected_idx_fit = np.asarray(selected_obj["S_idx"], dtype=int)
selected_eps_fit = float(selected_obj["eps"])
selected_pi_pos_fit = float(selected_obj["pi_pos"])
selected_P_pos_fit = np.asarray(selected_obj["P_pos"], dtype=float)


# ---------------------------------------------------------------
# 6) Selected support table with tickers and .3f confidence
# ---------------------------------------------------------------

map_order = np.argsort(-abs_w_map)
rank_abs_map = np.empty_like(map_order)
rank_abs_map[map_order] = np.arange(1, p + 1)

selected_mask = np.zeros(p, dtype=bool)
selected_mask[S_selected_idx_fit] = True

df_selected_support_fit = pd.DataFrame(
    {
        "j": S_selected_idx_fit,
        "ticker": [tickers_fit[j] for j in S_selected_idx_fit],
        "rank_abs_MAP": rank_abs_map[S_selected_idx_fit],
        "w_MAP": w_map[S_selected_idx_fit],
        "abs_w_MAP": abs_w_map[S_selected_idx_fit],
        "confidence_P_w_gt_eps": selected_P_pos_fit[S_selected_idx_fit],
        "confidence_margin": selected_P_pos_fit[S_selected_idx_fit] - selected_pi_pos_fit,
        "w_minus_eps": w_map[S_selected_idx_fit] - selected_eps_fit,
        "in_top50_MAP": rank_abs_map[S_selected_idx_fit] <= 50,
    }
)

df_selected_support_fit = (
    df_selected_support_fit
    .sort_values("rank_abs_MAP")
    .reset_index(drop=True)
)

n_in_top50 = int(np.sum(df_selected_support_fit["in_top50_MAP"].to_numpy()))
n_outside_top50 = int(len(df_selected_support_fit) - n_in_top50)

print(
    f"\nSelected support diagnostic: {candidate_name_fit}\n"
    f"Rule: w_MAP > {selected_eps_fit:.1e}, "
    f"P(w_j > eps | y) >= {selected_pi_pos_fit:.2f}\n"
    f"|S| = {len(S_selected_idx_fit)} | "
    f"in top-50 MAP = {n_in_top50} | "
    f"outside top-50 MAP = {n_outside_top50}"
)

df_selected_display = df_selected_support_fit.copy()
df_selected_display["w_MAP"] = df_selected_display["w_MAP"].map(lambda x: f"{x:+.4e}")
df_selected_display["abs_w_MAP"] = df_selected_display["abs_w_MAP"].map(lambda x: f"{x:.4e}")
df_selected_display["confidence_P_w_gt_eps"] = df_selected_display["confidence_P_w_gt_eps"].map(lambda x: f"{x:.3f}")
df_selected_display["confidence_margin"] = df_selected_display["confidence_margin"].map(lambda x: f"{x:+.3f}")
df_selected_display["w_minus_eps"] = df_selected_display["w_minus_eps"].map(lambda x: f"{x:+.4e}")

print("\n=== Selected support, ranked by |MAP| ===")
print(df_selected_display.to_string(index=False))


# ---------------------------------------------------------------
# 7) The 17 selected names outside the top-50 MAP
# ---------------------------------------------------------------

df_selected_outside_top50_fit = df_selected_support_fit[
    df_selected_support_fit["rank_abs_MAP"] > 50
].copy()

df_selected_outside_top50_fit = (
    df_selected_outside_top50_fit
    .sort_values("confidence_P_w_gt_eps", ascending=False)
    .reset_index(drop=True)
)

df_selected_outside_display = df_selected_outside_top50_fit.copy()
df_selected_outside_display["w_MAP"] = df_selected_outside_display["w_MAP"].map(lambda x: f"{x:+.4e}")
df_selected_outside_display["abs_w_MAP"] = df_selected_outside_display["abs_w_MAP"].map(lambda x: f"{x:.4e}")
df_selected_outside_display["confidence_P_w_gt_eps"] = df_selected_outside_display["confidence_P_w_gt_eps"].map(lambda x: f"{x:.3f}")
df_selected_outside_display["confidence_margin"] = df_selected_outside_display["confidence_margin"].map(lambda x: f"{x:+.3f}")
df_selected_outside_display["w_minus_eps"] = df_selected_outside_display["w_minus_eps"].map(lambda x: f"{x:+.4e}")

print(
    "\n=== Selected despite being outside top-50 MAP "
    f"({len(df_selected_outside_top50_fit)} names) ==="
)
print(df_selected_outside_display.to_string(index=False))


# ---------------------------------------------------------------
# 8) Top-50 MAP names not selected
#
# This helps diagnose the opposite case:
# names ranked highly by |MAP| but excluded because either
#   - w_MAP <= eps, or
#   - P(w_j > eps | y) < pi_pos.
# ---------------------------------------------------------------

top50_mask = np.zeros(p, dtype=bool)
top50_mask[top50_map] = True

top50_not_selected_idx = np.where(top50_mask & (~selected_mask))[0]

drop_reasons = []
for j in top50_not_selected_idx:
    if w_map[j] <= selected_eps_fit:
        reason = "w_MAP <= eps"
    elif selected_P_pos_fit[j] < selected_pi_pos_fit:
        reason = "confidence < pi_pos"
    else:
        reason = "other"

    drop_reasons.append(reason)

df_top50_not_selected_fit = pd.DataFrame(
    {
        "j": top50_not_selected_idx,
        "ticker": [tickers_fit[j] for j in top50_not_selected_idx],
        "rank_abs_MAP": rank_abs_map[top50_not_selected_idx],
        "w_MAP": w_map[top50_not_selected_idx],
        "abs_w_MAP": abs_w_map[top50_not_selected_idx],
        "confidence_P_w_gt_eps": selected_P_pos_fit[top50_not_selected_idx],
        "confidence_shortfall": selected_pi_pos_fit - selected_P_pos_fit[top50_not_selected_idx],
        "w_minus_eps": w_map[top50_not_selected_idx] - selected_eps_fit,
        "drop_reason": drop_reasons,
    }
)

df_top50_not_selected_fit = (
    df_top50_not_selected_fit
    .sort_values("rank_abs_MAP")
    .reset_index(drop=True)
)

df_top50_not_selected_display = df_top50_not_selected_fit.copy()
df_top50_not_selected_display["w_MAP"] = df_top50_not_selected_display["w_MAP"].map(lambda x: f"{x:+.4e}")
df_top50_not_selected_display["abs_w_MAP"] = df_top50_not_selected_display["abs_w_MAP"].map(lambda x: f"{x:.4e}")
df_top50_not_selected_display["confidence_P_w_gt_eps"] = df_top50_not_selected_display["confidence_P_w_gt_eps"].map(lambda x: f"{x:.3f}")
df_top50_not_selected_display["confidence_shortfall"] = df_top50_not_selected_display["confidence_shortfall"].map(lambda x: f"{x:+.3f}")
df_top50_not_selected_display["w_minus_eps"] = df_top50_not_selected_display["w_minus_eps"].map(lambda x: f"{x:+.4e}")

print(
    "\n=== Top-50 MAP names not selected "
    f"({len(df_top50_not_selected_fit)} names) ==="
)
print(df_top50_not_selected_display.to_string(index=False))


# ---------------------------------------------------------------
# 9) Save CSVs and bundle for next block
# ---------------------------------------------------------------

OUTDIR_SUPPORT = Path(f"./tables_practical_support_FIT{fit_k_current:02d}")
OUTDIR_SUPPORT.mkdir(parents=True, exist_ok=True)

df_practical_frontier.to_csv(
    OUTDIR_SUPPORT / "compact_practical_positive_frontier.csv",
    index=False,
)

df_selected_support_fit.to_csv(
    OUTDIR_SUPPORT / f"selected_support_{candidate_name_fit}.csv",
    index=False,
)

df_selected_outside_top50_fit.to_csv(
    OUTDIR_SUPPORT / f"selected_outside_top50_{candidate_name_fit}.csv",
    index=False,
)

df_top50_not_selected_fit.to_csv(
    OUTDIR_SUPPORT / f"top50_not_selected_{candidate_name_fit}.csv",
    index=False,
)

selected_support_fit = {
    "fit_k": fit_k_current,
    "candidate_name": candidate_name_fit,
    "eps": selected_eps_fit,
    "pi_pos": selected_pi_pos_fit,
    "S_idx": S_selected_idx_fit,
    "df_selected_support": df_selected_support_fit,
    "df_selected_outside_top50": df_selected_outside_top50_fit,
    "df_top50_not_selected": df_top50_not_selected_fit,
    "n_in_top50": n_in_top50,
    "n_outside_top50": n_outside_top50,
    "mass_kept": map_l1_mass_share(S_selected_idx_fit),
}

practical_support_summary_fit = {
    "df_practical_frontier": df_practical_frontier,
    "candidate_rules": candidate_rules,
    "candidate_aliases": candidate_aliases,
    "practical_support_candidates_fit": practical_support_candidates_fit,
    "selected_support_fit": selected_support_fit,
    "P_pos_eps": P_pos_eps,
}

print(f"\nSaved support diagnostic CSVs to: {OUTDIR_SUPPORT.resolve()}")
print("[summary] practical_support_candidates_fit created.")
print("[summary] selected_support_fit created for the next refit block.")

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — Block 11
# Long-only FISTA refit on selected posterior-informed support
#
# Purpose:
#   Take the selected support from Block 10b and compute the
#   long-only refitted portfolio on that support.
#
#   Main case-study default:
#       selected support = broad_eps1e-3_pi0.60
#
# This block:
#   - uses S_selected_idx_fit from Block 10b;
#   - refits only on S_selected_idx_fit;
#   - enforces w >= 0 via the proximal step;
#   - applies a final simplex projection on S so that
#         w >= 0 and sum_j w_j = 1 exactly;
#   - compares raw FIT tracking error against the untouched MAP;
#   - stores w_refit_fit and w_locked_fit for later blocks.
#
# Inputs assumed:
#   - selected_support_fit or S_selected_idx_fit
#   - practical_support_candidates_fit
#   - w_map_fit
#   - y_c_fit, R_c_fit, alpha_fit
#   - y_raw_fit, R_raw_fit, y_mu_fit, R_mu_fit
#   - sigma2_fit, Lambda_fit, theta_EB_fit
#   - te_rmse_raw_centered
#
# Optional:
#   - tickers / asset names recovered from notebook variables
# ===============================================================

from typing import Dict, Any, Optional, Tuple, List
import numpy as np
import pandas as pd


# ---------------------------------------------------------------
# 0) Select support rule
# ---------------------------------------------------------------

fit_k_current = int(globals().get("fit_k", 1))

# Main case-study choice.
candidate_name_fit = "broad_eps1e-3_pi0.60"

# Sensitivity alternatives kept for later:
# candidate_name_fit = "middle_eps2p5e-3_pi0.60"
# candidate_name_fit = "conservative_eps5e-3_pi0.60"


def _recover_selected_support(
    candidate_name: str,
) -> Dict[str, Any]:
    """
    Recover the selected support from the objects created in Block 10b.

    Priority:
      1. practical_support_candidates_fit[candidate_name]
      2. selected_support_fit
      3. S_selected_idx_fit
    """
    if (
        "practical_support_candidates_fit" in globals()
        and isinstance(practical_support_candidates_fit, dict)
        and candidate_name in practical_support_candidates_fit
    ):
        obj = practical_support_candidates_fit[candidate_name].copy()
        obj["candidate_name"] = candidate_name
        return obj

    if "selected_support_fit" in globals() and isinstance(selected_support_fit, dict):
        obj = selected_support_fit.copy()
        obj.setdefault("candidate_name", obj.get("candidate_name", candidate_name))
        return obj

    if "S_selected_idx_fit" in globals():
        return {
            "candidate_name": candidate_name,
            "S_idx": np.asarray(S_selected_idx_fit, dtype=int),
            "eps": np.nan,
            "pi_pos": np.nan,
            "mass_kept": np.nan,
        }

    raise KeyError(
        "Could not find a selected support. Run Block 10b first, or define "
        "S_selected_idx_fit manually."
    )


selected_support_for_refit_fit = _recover_selected_support(candidate_name_fit)

S_refit_idx_fit = np.asarray(
    selected_support_for_refit_fit["S_idx"],
    dtype=int,
)

if S_refit_idx_fit.size == 0:
    raise ValueError("Selected support is empty; cannot refit.")

print("\n[Block 11] Selected support for refit")
print(f"FIT window = FIT-{fit_k_current:02d}")
print(f"candidate_name = {selected_support_for_refit_fit.get('candidate_name', candidate_name_fit)}")
print(f"|S_refit| = {S_refit_idx_fit.size}")
print(f"eps = {selected_support_for_refit_fit.get('eps', np.nan):.1e}")
print(f"pi_pos = {selected_support_for_refit_fit.get('pi_pos', np.nan):.2f}")
print(f"MAP mass kept = {selected_support_for_refit_fit.get('mass_kept', np.nan):.3f}")


# ---------------------------------------------------------------
# 1) Optional ticker recovery
# ---------------------------------------------------------------

def get_optional_tickers_for_refit(p: int) -> List[str]:
    """
    Try common ticker variable names.

    Falls back to df_ret convention:
      - asset columns first;
      - index column named idx_name, or last column if idx_name absent.
    """
    candidate_names = [
        "asset_names",
        "tickers",
        "asset_cols",
        "asset_columns",
        "asset_names_fit",
        "tickers_fit",
    ]

    for name in candidate_names:
        if name in globals():
            val = list(globals()[name])
            if len(val) == p:
                return [str(x) for x in val]

    if "df_ret" in globals():
        cols = list(df_ret.columns)

        if "idx_name" in globals() and idx_name in cols:
            asset_cols_from_df = [c for c in cols if c != idx_name]
        else:
            asset_cols_from_df = cols[:-1]

        if len(asset_cols_from_df) == p:
            return [str(x) for x in asset_cols_from_df]

    return [f"asset_{j}" for j in range(p)]


p_fit = int(w_map_fit.size)
tickers_refit_fit = get_optional_tickers_for_refit(p_fit)


# ---------------------------------------------------------------
# 2) Proximal and numerical helpers
# ---------------------------------------------------------------

def prox_weighted_l1_nonneg(
    x: np.ndarray,
    ttheta: float,
    alpha: np.ndarray,
) -> np.ndarray:
    """
    Proximal operator of

        ttheta * sum_j alpha_j |w_j| + I{w >= 0}.

    Since the constraint is w >= 0, this is positive soft-thresholding:

        prox_j(x) = max(x_j - ttheta * alpha_j, 0).
    """
    x = np.asarray(x, dtype=float)
    alpha = np.asarray(alpha, dtype=float)

    return np.maximum(x - float(ttheta) * alpha, 0.0)


def project_to_simplex(
    v: np.ndarray,
    target: float = 1.0,
) -> np.ndarray:
    """
    Euclidean projection onto the simplex

        {x : x >= 0, sum_j x_j = target}.

    This is used as the final exact-budget correction.
    """
    v = np.asarray(v, dtype=float).reshape(-1)

    if v.size == 0:
        raise ValueError("Cannot project an empty vector onto the simplex.")

    if target <= 0:
        raise ValueError("Simplex target must be positive.")

    u = np.sort(v)[::-1]
    cssv = np.cumsum(u) - target
    rho_candidates = u - cssv / (np.arange(v.size) + 1) > 0

    if not np.any(rho_candidates):
        return np.ones(v.size) * target / v.size

    rho = np.where(rho_candidates)[0][-1]
    theta = cssv[rho] / float(rho + 1)

    w = np.maximum(v - theta, 0.0)

    # Small numerical correction.
    s = float(w.sum())
    if s > 0:
        w *= target / s

    return w


def _A_S_matvec_fit(
    vS: np.ndarray,
    R_S: np.ndarray,
    sigma2: float,
    Lambda: float,
) -> np.ndarray:
    """
    Matrix-vector product for the smooth Hessian on S:

        R_S^T R_S / sigma2 + 2 Lambda 11^T.
    """
    RtRv = R_S.T @ (R_S @ vS)
    budget_part = 2.0 * float(Lambda) * np.sum(vS) * np.ones_like(vS)

    return RtRv / float(sigma2) + budget_part


def power_iteration_on_S_fit(
    R_S: np.ndarray,
    sigma2: float,
    Lambda: float,
    *,
    max_iter: int = 300,
    tol: float = 1e-8,
    rng: Optional[np.random.Generator] = None,
) -> float:
    """
    Estimate the Lipschitz constant on the active set S.
    """
    rng = np.random.default_rng(2031) if rng is None else rng

    s = R_S.shape[1]
    if s == 0:
        raise ValueError("Cannot run power iteration on an empty active set.")

    v = rng.standard_normal(s)
    v /= np.linalg.norm(v) + 1e-15

    prev_val = 0.0
    val = 0.0

    for _ in range(max_iter):
        Av = _A_S_matvec_fit(v, R_S, sigma2, Lambda)
        val = float(np.dot(v, Av))

        norm_Av = np.linalg.norm(Av)
        if norm_Av <= 1e-15:
            break

        v_next = Av / norm_Av

        if abs(val - prev_val) <= tol * max(1.0, abs(val)):
            break

        v = v_next
        prev_val = val

    return float(max(val, 1e-15))


def objective_on_S_fit(
    y_c: np.ndarray,
    R_S: np.ndarray,
    w_S: np.ndarray,
    alpha_S: np.ndarray,
    sigma2: float,
    Lambda: float,
    theta: float,
) -> float:
    """
    Smoothed objective on active set S, including the weighted l1 term.
    """
    resid = y_c - R_S @ w_S

    data_term = 0.5 / float(sigma2) * float(resid @ resid)
    budget_term = float(Lambda) * float((np.sum(w_S) - 1.0) ** 2)
    penalty_term = float(theta) * float(np.sum(alpha_S * np.abs(w_S)))

    return float(data_term + budget_term + penalty_term)


# ---------------------------------------------------------------
# 3) FISTA refit on selected S
# ---------------------------------------------------------------

def fista_refit_selected_support_nonneg_fit(
    *,
    y_c: np.ndarray,
    R_c: np.ndarray,
    alpha: np.ndarray,
    sigma2: float,
    Lambda: float,
    theta: float,
    S_idx: np.ndarray,
    L_S: Optional[float] = None,
    max_iter: int = 5000,
    tol: float = 1e-7,
    simplex_final: bool = True,
    rng: Optional[np.random.Generator] = None,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    """
    Long-only FISTA refit on a fixed active support S:

        min_w  0.5/sigma2 ||y_c - R_S w||^2
             + Lambda (1^T w - 1)^2
             + theta sum_{j in S} alpha_j |w_j|

        subject to w_j >= 0 for j in S.

    The returned vector is full p-dimensional, with zeros outside S.

    If simplex_final=True, the final active vector is projected to
    {w >= 0, sum(w)=1}. This makes the output immediately tradeable.
    """
    y_c = np.asarray(y_c, dtype=float).reshape(-1)
    R_c = np.asarray(R_c, dtype=float)
    alpha = np.asarray(alpha, dtype=float).reshape(-1)
    S = np.asarray(S_idx, dtype=int)

    if S.size == 0:
        raise ValueError("S_idx is empty.")

    if R_c.shape[1] != alpha.size:
        raise ValueError("R_c and alpha have incompatible dimensions.")

    p = R_c.shape[1]
    R_S = R_c[:, S]
    alpha_S = alpha[S]
    one_S = np.ones(S.size)

    if L_S is None:
        L_S = power_iteration_on_S_fit(
            R_S=R_S,
            sigma2=sigma2,
            Lambda=Lambda,
            rng=rng,
        )

    step = 1.0 / float(L_S)

    # Start from an equal-weight vector on S. This is already long-only
    # and budget-feasible, which is usually a good stabilising start.
    w = np.ones(S.size) / S.size
    z = w.copy()
    t = 1.0

    obj_trace = []
    sum_trace = []
    nnz_trace = []

    prev_obj = objective_on_S_fit(
        y_c=y_c,
        R_S=R_S,
        w_S=w,
        alpha_S=alpha_S,
        sigma2=sigma2,
        Lambda=Lambda,
        theta=theta,
    )

    obj_trace.append(prev_obj)
    sum_trace.append(float(np.sum(w)))
    nnz_trace.append(int(np.count_nonzero(w)))

    it_last = 0

    for it in range(1, max_iter + 1):
        resid_z = R_S @ z - y_c

        grad = (
            (R_S.T @ resid_z) / float(sigma2)
            + 2.0 * float(Lambda) * (np.sum(z) - 1.0) * one_S
        )

        w_next = prox_weighted_l1_nonneg(
            x=z - step * grad,
            ttheta=step * float(theta),
            alpha=alpha_S,
        )

        obj = objective_on_S_fit(
            y_c=y_c,
            R_S=R_S,
            w_S=w_next,
            alpha_S=alpha_S,
            sigma2=sigma2,
            Lambda=Lambda,
            theta=theta,
        )

        # Standard FISTA acceleration.
        t_next = 0.5 * (1.0 + np.sqrt(1.0 + 4.0 * t * t))
        z_next = w_next + ((t - 1.0) / t_next) * (w_next - w)

        obj_trace.append(obj)
        sum_trace.append(float(np.sum(w_next)))
        nnz_trace.append(int(np.count_nonzero(w_next)))

        if abs(prev_obj - obj) <= tol * max(1.0, abs(prev_obj)):
            w = w_next
            it_last = it
            break

        w = w_next
        z = z_next
        t = t_next
        prev_obj = obj
        it_last = it

    w_before_simplex = w.copy()

    if simplex_final:
        w = project_to_simplex(w, target=1.0)

    w_full = np.zeros(p)
    w_full[S] = w

    w_before_full = np.zeros(p)
    w_before_full[S] = w_before_simplex

    trace = {
        "obj": np.asarray(obj_trace),
        "sum_w_S": np.asarray(sum_trace),
        "nnz_S": np.asarray(nnz_trace),
        "iters": int(it_last),
        "L_S": float(L_S),
        "step": float(step),
        "simplex_final": bool(simplex_final),
        "w_before_simplex": w_before_full,
        "sum_before_simplex": float(np.sum(w_before_simplex)),
        "nnz_before_simplex": int(np.count_nonzero(w_before_simplex)),
    }

    return w_full, trace


w_refit_fit, refit_trace_fit = fista_refit_selected_support_nonneg_fit(
    y_c=y_c_fit,
    R_c=R_c_fit,
    alpha=alpha_fit,
    sigma2=sigma2_fit,
    Lambda=Lambda_fit,
    theta=theta_EB_fit,
    S_idx=S_refit_idx_fit,
    L_S=None,
    max_iter=5000,
    tol=1e-7,
    simplex_final=True,
    rng=np.random.default_rng(2031),
)

# The current locked FIT portfolio for downstream hold-period evaluation.
w_locked_fit = w_refit_fit
S_locked_idx_fit = S_refit_idx_fit.copy()


# ---------------------------------------------------------------
# 4) Tracking-error comparison on the FIT window
# ---------------------------------------------------------------

te_map_fit = te_rmse_raw_centered(
    y_raw=y_raw_fit,
    R_raw=R_raw_fit,
    w=w_map_fit,
    y_mu=y_mu_fit,
    R_mu=R_mu_fit,
)

te_refit_fit = te_rmse_raw_centered(
    y_raw=y_raw_fit,
    R_raw=R_raw_fit,
    w=w_refit_fit,
    y_mu=y_mu_fit,
    R_mu=R_mu_fit,
)

te_before_simplex_fit = te_rmse_raw_centered(
    y_raw=y_raw_fit,
    R_raw=R_raw_fit,
    w=refit_trace_fit["w_before_simplex"],
    y_mu=y_mu_fit,
    R_mu=R_mu_fit,
)

pred_gap_map_vs_refit_fit = float(
    np.sqrt(np.mean((R_c_fit @ (w_map_fit - w_refit_fit)) ** 2))
)

print("\n[Block 11] FIT-window refit summary")
print(f"support rule = {selected_support_for_refit_fit.get('candidate_name', candidate_name_fit)}")
print(f"|S_refit| = {S_refit_idx_fit.size}")
print(f"FISTA iterations = {refit_trace_fit['iters']}")
print(f"L_S = {refit_trace_fit['L_S']:.6e}")
print(f"step = {refit_trace_fit['step']:.6e}")
print(f"sum before simplex = {refit_trace_fit['sum_before_simplex']:+.12f}")
print(f"nnz before simplex = {refit_trace_fit['nnz_before_simplex']}")
print(f"sum after simplex  = {w_refit_fit.sum():+.12f}")
print(f"nnz after simplex  = {np.count_nonzero(w_refit_fit)} / {w_refit_fit.size}")
print(f"negative weights after simplex = {np.count_nonzero(w_refit_fit < 0.0)}")
print(f"TE(raw|FIT) MAP              = {te_map_fit:.6e}")
print(f"TE(raw|FIT) refit before simplex = {te_before_simplex_fit:.6e}")
print(f"TE(raw|FIT) refit after simplex  = {te_refit_fit:.6e}")
print(f"RMS prediction gap MAP vs refit  = {pred_gap_map_vs_refit_fit:.6e}")


# ---------------------------------------------------------------
# 5) Refit holding table
# ---------------------------------------------------------------

def build_refit_weight_table_fit(
    *,
    S_idx: np.ndarray,
    w_map: np.ndarray,
    w_refit: np.ndarray,
    tickers: List[str],
    selected_obj: Dict[str, Any],
) -> pd.DataFrame:
    """
    Build a readable table of selected/refitted holdings.
    """
    S = np.asarray(S_idx, dtype=int)

    # Sort by refitted weight, descending.
    S_ordered = S[np.argsort(-w_refit[S])]

    P_pos = selected_obj.get("P_pos", None)
    if P_pos is None:
        P_pos_vals = np.full(S_ordered.size, np.nan)
    else:
        P_pos_vals = np.asarray(P_pos, dtype=float)[S_ordered]

    df = pd.DataFrame(
        {
            "rank_refit": np.arange(1, S_ordered.size + 1),
            "j": S_ordered,
            "ticker": [tickers[j] for j in S_ordered],
            "w_refit": w_refit[S_ordered],
            "w_MAP": w_map[S_ordered],
            "delta_refit_minus_MAP": w_refit[S_ordered] - w_map[S_ordered],
            "P_w_gt_eps": P_pos_vals,
        }
    )

    return df


df_refit_weights_fit = build_refit_weight_table_fit(
    S_idx=S_refit_idx_fit,
    w_map=w_map_fit,
    w_refit=w_refit_fit,
    tickers=tickers_refit_fit,
    selected_obj=selected_support_for_refit_fit,
)

df_refit_display_fit = df_refit_weights_fit.copy()
for col in ["w_refit", "w_MAP", "delta_refit_minus_MAP"]:
    df_refit_display_fit[col] = df_refit_display_fit[col].map(lambda x: f"{x:+.6e}")

df_refit_display_fit["P_w_gt_eps"] = df_refit_display_fit["P_w_gt_eps"].map(
    lambda x: "" if pd.isna(x) else f"{float(x):.3f}"
)

print("\n=== Refit weights on selected support, sorted by refitted weight ===")
print(df_refit_display_fit.to_string(index=False))


# ---------------------------------------------------------------
# 6) Bundle for downstream blocks
# ---------------------------------------------------------------

refit_selected_support_fit = {
    "fit_k": fit_k_current,
    "candidate_name": selected_support_for_refit_fit.get("candidate_name", candidate_name_fit),
    "eps": selected_support_for_refit_fit.get("eps", np.nan),
    "pi_pos": selected_support_for_refit_fit.get("pi_pos", np.nan),
    "mass_kept": selected_support_for_refit_fit.get("mass_kept", np.nan),

    # Support and weights.
    "S_idx": S_refit_idx_fit,
    "S_refit_idx": S_refit_idx_fit,
    "S_locked_idx": S_locked_idx_fit,
    "w_map": w_map_fit,
    "w_refit_before_simplex": refit_trace_fit["w_before_simplex"],
    "w_refit_after_simplex": w_refit_fit,
    "w_refit": w_refit_fit,
    "w_locked_fit": w_locked_fit,

    # Solver diagnostics, exposed at top level for reporting drivers.
    "fista_iterations": int(refit_trace_fit["iters"]),
    "fista_iter": int(refit_trace_fit["iters"]),
    "L_S": float(refit_trace_fit["L_S"]),
    "step": float(refit_trace_fit["step"]),
    "step_size": float(refit_trace_fit["step"]),
    "simplex_final": bool(refit_trace_fit["simplex_final"]),
    "sum_before_simplex": float(refit_trace_fit["sum_before_simplex"]),
    "nnz_before_simplex": int(refit_trace_fit["nnz_before_simplex"]),
    "sum_after_simplex": float(w_refit_fit.sum()),
    "nnz_after_simplex": int(np.count_nonzero(w_refit_fit)),
    "negative_weights_after_simplex": int(np.count_nonzero(w_refit_fit < 0.0)),

    # TE diagnostics.
    "te_fit": {
        "map": float(te_map_fit),
        "refit_before_simplex": float(te_before_simplex_fit),
        "refit_after_simplex": float(te_refit_fit),
    },
    "TE_MAP": float(te_map_fit),
    "TE_refit_before_simplex": float(te_before_simplex_fit),
    "TE_refit_after_simplex": float(te_refit_fit),
    "pred_gap_map_vs_refit": float(pred_gap_map_vs_refit_fit),
    "rms_prediction_gap": float(pred_gap_map_vs_refit_fit),

    # Tables and raw trace.
    "df_refit_weights": df_refit_weights_fit,
    "selected_support": selected_support_for_refit_fit,
    "trace": refit_trace_fit,
}

print("\n[Block 11] refit_selected_support_fit created.")
print("[Block 11] w_locked_fit and S_locked_idx_fit are ready for hold-period evaluation.")

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — Block 11B
# One-off implementability floor before Hold-1 evaluation/export
#
# Purpose:
#   Apply the construction-stage floor that is used in the final ASOC
#   case-study branch before evaluating the first holding window.
#
# Important:
#   This is a one-off construction/export adjustment.  It is not part
#   of the sequential rebalancing rule and it is not applied repeatedly.
#   The full-horizon notebook reads the metadata below and will not apply
#   the same initial floor again unless explicitly forced.
# ===============================================================

from typing import Dict, Any
import numpy as np
import pandas as pd


# ---------------------------------------------------------------
# User-facing switches
# ---------------------------------------------------------------

APPLY_ONE_OFF_INITIAL_FLOOR = bool(APPLY_INITIAL_FLOOR)
ONE_OFF_INITIAL_FLOOR = float(INITIAL_FLOOR_WEIGHT)
ONE_OFF_INITIAL_REALLOC = str(INITIAL_FLOOR_REALLOC)
ONE_OFF_INITIAL_FLOOR_TOL = 1e-12


# ---------------------------------------------------------------
# Small diagnostics and floor helper
# ---------------------------------------------------------------

def _asoc_weight_structure_for_logs(w, *, tol=1e-12) -> Dict[str, Any]:
    """Return a compact portfolio-structure summary for floor logs."""
    w = np.asarray(w, dtype=float).reshape(-1)
    abs_w = np.abs(w)
    l1 = float(abs_w.sum())
    pos = w[w > tol]
    order = np.argsort(-abs_w)

    return {
        "sum_w": float(w.sum()),
        "nnz": int(np.count_nonzero(abs_w > tol)),
        "nnz_ge_0p5pct": int(np.count_nonzero(w >= 0.005 - tol)),
        "nnz_ge_1pct": int(np.count_nonzero(w >= 0.010 - tol)),
        "n_negative": int(np.count_nonzero(w < -tol)),
        "min_positive": float(np.min(pos)) if pos.size else np.nan,
        "top10_L1_share": float(abs_w[order[:10]].sum() / l1) if l1 > 0 else np.nan,
        "top25_L1_share": float(abs_w[order[:25]].sum() / l1) if l1 > 0 else np.nan,
        "HHI_abs": float(np.sum((abs_w / l1) ** 2)) if l1 > 0 else np.nan,
    }


def apply_min_weight_floor_once_logged(
    w,
    asset_cols,
    *,
    floor,
    hold,
    variant_label,
    action_stage="initial_prune",
    realloc="proportional_survivors",
    tol=1e-12,
):
    """
    Prune positive holdings below `floor` and reallocate their mass.

    The redistribution is proportional over surviving positive holdings.
    This is used only once, before the first holding-window evaluation.
    """
    w_old = np.asarray(w, dtype=float).reshape(-1)
    w_new = w_old.copy()
    asset_cols = list(asset_cols)
    floor = float(floor)
    rows = []

    if len(asset_cols) != w_new.size:
        raise ValueError(
            "asset_cols length does not match weight dimension: "
            f"len(asset_cols)={len(asset_cols)}, p={w_new.size}."
        )

    before = _asoc_weight_structure_for_logs(w_new, tol=tol)

    prune_mask = (w_new > tol) & (w_new < floor)
    prune_idx = np.where(prune_mask)[0]
    mass = float(w_new[prune_idx].sum())

    for j in prune_idx:
        rows.append({
            "variant": variant_label,
            "hold": int(hold),
            "action_stage": action_stage,
            "ticker": asset_cols[j],
            "old_weight": float(w_old[j]),
            "new_weight": 0.0,
            "delta_weight": float(-w_old[j]),
            "reason": "below_floor_pruned",
            "floor": floor,
            "realloc": realloc,
            "mass_pruned_total": mass,
        })
        w_new[j] = 0.0

    if mass > tol:
        survivor_idx = np.where(w_new > tol)[0]
        if survivor_idx.size == 0:
            raise ValueError("No surviving holdings available to receive pruned mass.")

        if realloc != "proportional_survivors":
            raise ValueError(
                f"Unsupported realloc={realloc!r}; only proportional_survivors is implemented."
            )

        denom = float(w_new[survivor_idx].sum())
        inc = mass * w_new[survivor_idx] / denom
        old_survivor_weights = w_new[survivor_idx].copy()
        w_new[survivor_idx] += inc

        for j, old_j, inc_j in zip(survivor_idx, old_survivor_weights, inc):
            if abs(float(inc_j)) > tol:
                rows.append({
                    "variant": variant_label,
                    "hold": int(hold),
                    "action_stage": action_stage,
                    "ticker": asset_cols[j],
                    "old_weight": float(old_j),
                    "new_weight": float(w_new[j]),
                    "delta_weight": float(inc_j),
                    "reason": "redistributed_pruned_mass_proportional_survivors",
                    "floor": floor,
                    "realloc": realloc,
                    "mass_pruned_total": mass,
                })

    # Local numerical budget correction on the largest survivor only.
    residual = float(w_old.sum() - w_new.sum())
    if abs(residual) > 1e-14:
        survivor_idx = np.where(w_new > tol)[0]
        if survivor_idx.size == 0:
            raise ValueError("No survivor available for numerical budget correction.")
        j_fix = int(survivor_idx[np.argmax(w_new[survivor_idx])])
        old_j = float(w_new[j_fix])
        w_new[j_fix] += residual
        rows.append({
            "variant": variant_label,
            "hold": int(hold),
            "action_stage": action_stage,
            "ticker": asset_cols[j_fix],
            "old_weight": old_j,
            "new_weight": float(w_new[j_fix]),
            "delta_weight": residual,
            "reason": "local_numerical_budget_correction",
            "floor": floor,
            "realloc": realloc,
            "mass_pruned_total": mass,
        })

    w_new[np.abs(w_new) < 1e-15] = 0.0
    after = _asoc_weight_structure_for_logs(w_new, tol=tol)

    print("\n[One-off initial floor action]")
    print(f"variant          = {variant_label}")
    print(f"floor            = {floor:.4f}")
    print(f"pruned names     = {len(prune_idx)}")
    print(f"mass pruned      = {mass:.6e}")
    print(f"sum before/after = {before['sum_w']:.12f} / {after['sum_w']:.12f}")
    print(f"nnz before/after = {before['nnz']} / {after['nnz']}")
    print(f"min pos before   = {before['min_positive']:.6e}")
    print(f"min pos after    = {after['min_positive']:.6e}")
    print(f"one-way turnover = {0.5 * np.sum(np.abs(w_new - w_old)):.6e}")

    return w_new, pd.DataFrame(rows), {"before": before, "after": after, "mass_pruned": mass}


# ---------------------------------------------------------------
# Apply floor to the locked refitted Tracker
# ---------------------------------------------------------------

if "w_locked_fit" not in globals():
    raise RuntimeError("Run Block 11 before Block 11B: w_locked_fit is missing.")

if "idx_name" not in globals():
    raise RuntimeError("idx_name is missing; cannot recover asset columns safely.")

asset_cols_for_floor_fit = [str(c) for c in df_ret.columns if str(c) != str(idx_name)]

w_locked_fit_pre_floor = np.asarray(w_locked_fit, dtype=float).reshape(-1).copy()
S_locked_idx_pre_floor = np.asarray(S_locked_idx_fit, dtype=int).reshape(-1).copy()

if len(asset_cols_for_floor_fit) != w_locked_fit_pre_floor.size:
    raise ValueError(
        "Asset-column count does not match w_locked_fit length before floor: "
        f"len(asset_cols_for_floor_fit)={len(asset_cols_for_floor_fit)}, "
        f"p={w_locked_fit_pre_floor.size}."
    )

initial_floor_actions_fit = pd.DataFrame()
initial_floor_diagnostics_fit = {
    "before": _asoc_weight_structure_for_logs(w_locked_fit_pre_floor),
    "after": _asoc_weight_structure_for_logs(w_locked_fit_pre_floor),
    "mass_pruned": 0.0,
}

if APPLY_ONE_OFF_INITIAL_FLOOR:
    w_locked_fit, initial_floor_actions_fit, initial_floor_diagnostics_fit = apply_min_weight_floor_once_logged(
        w_locked_fit_pre_floor,
        asset_cols_for_floor_fit,
        floor=float(ONE_OFF_INITIAL_FLOOR),
        hold=1,
        variant_label="initial_tracker_export",
        action_stage="initial_floor_before_hold1",
        realloc=ONE_OFF_INITIAL_REALLOC,
        tol=ONE_OFF_INITIAL_FLOOR_TOL,
    )
else:
    w_locked_fit = w_locked_fit_pre_floor.copy()
    print("\n[One-off initial floor action] skipped by APPLY_ONE_OFF_INITIAL_FLOOR=False")

S_locked_idx_fit = np.where(w_locked_fit > ONE_OFF_INITIAL_FLOOR_TOL)[0].astype(int)
INITIAL_FLOOR_APPLIED_TO_LOCKED_TRACKER = bool(APPLY_ONE_OFF_INITIAL_FLOOR)
INITIAL_FLOOR_VALUE_LOCKED_TRACKER = float(ONE_OFF_INITIAL_FLOOR) if APPLY_ONE_OFF_INITIAL_FLOOR else np.nan
INITIAL_FLOOR_REALLOC_LOCKED_TRACKER = str(ONE_OFF_INITIAL_REALLOC)
INITIAL_FLOOR_TURNOVER_FROM_REFIT = float(0.5 * np.sum(np.abs(w_locked_fit - w_locked_fit_pre_floor)))

# Keep refit bundle synchronized with the object that subsequent Hold-1 and export blocks use.
if isinstance(refit_selected_support_fit, dict):
    refit_selected_support_fit["w_locked_fit_pre_floor"] = w_locked_fit_pre_floor
    refit_selected_support_fit["S_locked_idx_pre_floor"] = S_locked_idx_pre_floor
    refit_selected_support_fit["w_locked_fit"] = w_locked_fit
    refit_selected_support_fit["S_locked_idx"] = S_locked_idx_fit
    refit_selected_support_fit["initial_floor_applied"] = INITIAL_FLOOR_APPLIED_TO_LOCKED_TRACKER
    refit_selected_support_fit["initial_floor_value"] = INITIAL_FLOOR_VALUE_LOCKED_TRACKER
    refit_selected_support_fit["initial_floor_realloc"] = INITIAL_FLOOR_REALLOC_LOCKED_TRACKER
    refit_selected_support_fit["initial_floor_turnover_from_refit"] = INITIAL_FLOOR_TURNOVER_FROM_REFIT
    refit_selected_support_fit["initial_floor_diagnostics"] = initial_floor_diagnostics_fit

print("\n[Block 11B] Locked Tracker after one-off floor is ready for Hold-1 evaluation/export.")
print(f"floor applied              = {INITIAL_FLOOR_APPLIED_TO_LOCKED_TRACKER}")
print(f"floor value                = {INITIAL_FLOOR_VALUE_LOCKED_TRACKER:.4f}")
print(f"support before/after       = {S_locked_idx_pre_floor.size} / {S_locked_idx_fit.size}")
print(f"sum before/after           = {w_locked_fit_pre_floor.sum():+.12f} / {w_locked_fit.sum():+.12f}")
print(f"one-way floor turnover     = {INITIAL_FLOOR_TURNOVER_FROM_REFIT:.6e}")
print(f"shorts after floor         = {np.count_nonzero(w_locked_fit < -1e-12)}")


In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — Block 12A
# Hold-window realised tracking metrics for the locked Tracker
#   Uses w_locked_fit after the optional one-off initial floor from Block 11B.
#
# Purpose:
#   Evaluate the locked held portfolio on Hold-1 using realised,
#   uncentred returns:
#
#       active_t = portfolio_return_t - index_return_t
#                = R_hold1[t, :] @ w_locked_fit - y_hold1[t].
#
#   This is the literature-comparable hold-window TE convention.
#
# Inputs assumed:
#   - w_locked_fit
#   - y_hold1, R_hold1, dates_h1
#
# Optional inputs:
#   - y_raw_fit, R_raw_fit
#       used to compute the FIT-window TE target for hit-rate reporting.
#
# Outputs:
#   - df_hold1_tracker_summary
#   - hold1_tracker_results
# ===============================================================

from pathlib import Path
from typing import Dict, Any, Optional

import numpy as np
import pandas as pd


# ---------------------------------------------------------------
# 0) Basic objects and checks
# ---------------------------------------------------------------

fit_k_current = int(globals().get("fit_k", 1))

TRACKER_LABEL = "Tracker"  # generic display label; support size is read from the generated cache

w_tracker = np.asarray(w_locked_fit, dtype=float).reshape(-1)
y_h1 = np.asarray(y_hold1, dtype=float).reshape(-1)
R_h1 = np.asarray(R_hold1, dtype=float)
dates_h1_pd = pd.to_datetime(dates_h1)

if R_h1.ndim != 2:
    raise ValueError("R_hold1 must be a 2D array.")

if R_h1.shape[0] != y_h1.size:
    raise ValueError(
        f"R_hold1 has {R_h1.shape[0]} rows, but y_hold1 has length {y_h1.size}."
    )

if R_h1.shape[1] != w_tracker.size:
    raise ValueError(
        f"R_hold1 has {R_h1.shape[1]} columns, but w_locked_fit has length {w_tracker.size}."
    )

if len(dates_h1_pd) != y_h1.size:
    raise ValueError(
        f"dates_h1 has length {len(dates_h1_pd)}, but y_hold1 has length {y_h1.size}."
    )


# ---------------------------------------------------------------
# 1) Realised-return helpers
# ---------------------------------------------------------------

def portfolio_returns_realised(
    R_raw: np.ndarray,
    w: np.ndarray,
) -> np.ndarray:
    """
    Realised portfolio returns R_raw @ w.

    No centering, no intercept, no fit-window mean correction.
    """
    R_raw = np.asarray(R_raw, dtype=float)
    w = np.asarray(w, dtype=float).reshape(-1)

    return (R_raw @ w).reshape(-1)


def active_returns_realised(
    y_raw: np.ndarray,
    R_raw: np.ndarray,
    w: np.ndarray,
) -> np.ndarray:
    """
    Realised active return series:

        active_t = portfolio_t - index_t.
    """
    y_raw = np.asarray(y_raw, dtype=float).reshape(-1)
    r_port = portfolio_returns_realised(R_raw, w)

    return r_port - y_raw


def te_rmse_realised(
    y_raw: np.ndarray,
    R_raw: np.ndarray,
    w: np.ndarray,
) -> float:
    """
    Realised RMSE tracking error.
    """
    active = active_returns_realised(y_raw, R_raw, w)

    return float(np.sqrt(np.mean(active ** 2)))


def rolling_rmse_realised(
    x: np.ndarray,
    window: int = 20,
) -> np.ndarray:
    """
    Rolling RMSE for a one-dimensional series.
    """
    x = np.asarray(x, dtype=float).reshape(-1)
    s = pd.Series(x)

    return (
        s.pow(2)
        .rolling(window, min_periods=window)
        .mean()
        .pow(0.5)
        .to_numpy()
    )


def drawdown_from_cumulative_active(
    cumulative_active: np.ndarray,
) -> tuple[float, np.ndarray]:
    """
    Drawdown of cumulative active return.
    """
    cumulative_active = np.asarray(cumulative_active, dtype=float).reshape(-1)

    running_max = np.maximum.accumulate(cumulative_active)
    drawdown = cumulative_active - running_max

    return float(drawdown.min()), drawdown


def structure_checks_tracker(
    w: np.ndarray,
    topk: tuple[int, ...] = (10, 25),
) -> Dict[str, float]:
    """
    Structure checks for the held portfolio at inception.
    """
    w = np.asarray(w, dtype=float).reshape(-1)

    L1 = float(np.sum(np.abs(w)) + 1e-16)
    order = np.argsort(-np.abs(w))

    out = {
        "sum_weights": float(w.sum()),
        "nnz": int(np.count_nonzero(w)),
        "n_shorts": int(np.count_nonzero(w < 0.0)),
        "HHI_abs": float(np.sum((np.abs(w) / L1) ** 2)),
    }

    for K in topk:
        out[f"top{K}_L1_share"] = float(np.sum(np.abs(w[order[:K]])) / L1)

    return out


# ---------------------------------------------------------------
# 2) Optional FIT-window TE target for hit-rate
# ---------------------------------------------------------------

def maybe_compute_fit_te_target_tracker() -> Optional[float]:
    """
    Compute realised FIT-window TE for Tracker if y_raw_fit and R_raw_fit exist.

    This is used as the hit-rate reference threshold:
        mean(|active_hold| < TE_fit_tracker).
    """
    if ("y_raw_fit" in globals()) and ("R_raw_fit" in globals()):
        return te_rmse_realised(
            y_raw=np.asarray(globals()["y_raw_fit"], dtype=float).reshape(-1),
            R_raw=np.asarray(globals()["R_raw_fit"], dtype=float),
            w=w_tracker,
        )

    return None


te_target_tracker_fit = maybe_compute_fit_te_target_tracker()


# ---------------------------------------------------------------
# 3) Hold-1 realised metrics
# ---------------------------------------------------------------

active_h1_tracker = active_returns_realised(
    y_raw=y_h1,
    R_raw=R_h1,
    w=w_tracker,
)

r_tracker_h1 = portfolio_returns_realised(R_h1, w_tracker)
r_index_h1 = y_h1.copy()

te_h1_tracker = float(np.sqrt(np.mean(active_h1_tracker ** 2)))
mae_h1_tracker = float(np.mean(np.abs(active_h1_tracker)))
mean_active_h1 = float(np.mean(active_h1_tracker))
std_active_h1 = float(np.std(active_h1_tracker, ddof=1))

rolling_te20_h1_tracker = rolling_rmse_realised(active_h1_tracker, window=20)

cum_active_h1_tracker = np.cumsum(active_h1_tracker)
max_active_drawdown_h1, drawdown_h1_tracker = drawdown_from_cumulative_active(
    cum_active_h1_tracker
)

if te_target_tracker_fit is not None:
    hit_rate_vs_fit_te = float(np.mean(np.abs(active_h1_tracker) < te_target_tracker_fit))
else:
    hit_rate_vs_fit_te = np.nan

struct_tracker = structure_checks_tracker(w_tracker, topk=(10, 25))

df_hold1_tracker_summary = pd.DataFrame(
    [
        {
            "portfolio": TRACKER_LABEL,
            "TE_RMSE": te_h1_tracker,
            "TE_bp": 1e4 * te_h1_tracker,
            "MAE_bp": 1e4 * mae_h1_tracker,
            "mean_active_bp": 1e4 * mean_active_h1,
            "std_active_bp": 1e4 * std_active_h1,
            "FIT_TE_target": te_target_tracker_fit,
            "FIT_TE_target_bp": (
                np.nan if te_target_tracker_fit is None else 1e4 * te_target_tracker_fit
            ),
            "hit_rate_vs_FIT_TE": hit_rate_vs_fit_te,
            "max_active_drawdown": max_active_drawdown_h1,
            **struct_tracker,
        }
    ]
)

print("\n=== Hold-1 realised summary: Tracker ===")
print(
    df_hold1_tracker_summary.to_string(
        index=False,
        formatters={
            "TE_RMSE": lambda x: f"{x:.6e}",
            "TE_bp": lambda x: f"{x:.3f}",
            "MAE_bp": lambda x: f"{x:.3f}",
            "mean_active_bp": lambda x: f"{x:+.3f}",
            "std_active_bp": lambda x: f"{x:.3f}",
            "FIT_TE_target": lambda x: "" if pd.isna(x) else f"{x:.6e}",
            "FIT_TE_target_bp": lambda x: "" if pd.isna(x) else f"{x:.3f}",
            "hit_rate_vs_FIT_TE": lambda x: "" if pd.isna(x) else f"{x:.3f}",
            "max_active_drawdown": lambda x: f"{x:+.6e}",
            "sum_weights": lambda x: f"{x:+.12f}",
            "HHI_abs": lambda x: f"{x:.6f}",
            "top10_L1_share": lambda x: f"{x:.3f}",
            "top25_L1_share": lambda x: f"{x:.3f}",
        },
    )
)

print("\nNotes:")
print(" • Hold-1 TE is computed from realised uncentred returns: R_hold1 @ w - y_hold1.")
print(" • 'Tracker' denotes the locked refitted portfolio from Block 11.")
print(" • Hit-rate uses the Tracker's realised FIT-window TE if y_raw_fit and R_raw_fit are available.")


# ---------------------------------------------------------------
# 4) Bundle for plotting/reporting blocks
# ---------------------------------------------------------------

hold1_tracker_results = {
    "fit_k": fit_k_current,
    "label": TRACKER_LABEL,
    "w": w_tracker,
    "dates": dates_h1_pd,
    "returns": {
        "index": r_index_h1,
        "tracker": r_tracker_h1,
        "active": active_h1_tracker,
    },
    "series": {
        "rolling_te20": rolling_te20_h1_tracker,
        "cum_active": cum_active_h1_tracker,
        "drawdown": drawdown_h1_tracker,
    },
    "scalars": {
        "TE_RMSE": te_h1_tracker,
        "TE_bp": 1e4 * te_h1_tracker,
        "MAE_bp": 1e4 * mae_h1_tracker,
        "mean_active_bp": 1e4 * mean_active_h1,
        "std_active_bp": 1e4 * std_active_h1,
        "FIT_TE_target": te_target_tracker_fit,
        "FIT_TE_target_bp": (
            np.nan if te_target_tracker_fit is None else 1e4 * te_target_tracker_fit
        ),
        "hit_rate_vs_FIT_TE": hit_rate_vs_fit_te,
        "max_active_drawdown": max_active_drawdown_h1,
        **struct_tracker,
    },
    "df_summary": df_hold1_tracker_summary,
}

print("\n[Block 12A] hold1_tracker_results created.")

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — Block 12B
# Hold-window realised figures for the locked Tracker
#
# Purpose:
#   Plot Hold-1 realised out-of-sample tracking diagnostics:
#     1. cumulative return overlay;
#     2. daily absolute active return in bp;
#     3. 20-day rolling RMSE tracking error in bp;
#     4. cumulative active return;
#     5. active-return histogram.
#
# Inputs assumed:
#   - hold1_tracker_results from Block 12A
#
# Outputs:
#   - figures saved to H1_DIR
# ===============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


# ---------------------------------------------------------------
# Figure directory
# ---------------------------------------------------------------

fit_k_current = int(hold1_tracker_results.get("fit_k", globals().get("fit_k", 1)))

H1_DIR = Path(OUTPUT_DIR) / "figures" / f"hold1_FIT{fit_k_current:02d}"
H1_DIR.mkdir(parents=True, exist_ok=True)

TRACKER_LABEL = hold1_tracker_results.get("label", "Tracker")


# ---------------------------------------------------------------
# Date-axis helper
# ---------------------------------------------------------------

def sparsify_date_axis(
    ax,
    *,
    minticks: int = 3,
    maxticks: int = 6,
) -> None:
    """
    Make date labels readable by using a sparse automatic date locator.

    This is especially useful for the 20-day rolling TE plot, where dense
    date labels can otherwise turn into a smudge.
    """
    locator = mdates.AutoDateLocator(minticks=minticks, maxticks=maxticks)
    formatter = mdates.ConciseDateFormatter(locator)

    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)
    ax.tick_params(axis="x", labelrotation=0)


# ---------------------------------------------------------------
# Extract realised series
# ---------------------------------------------------------------

dates = pd.to_datetime(hold1_tracker_results["dates"])

r_index = np.asarray(hold1_tracker_results["returns"]["index"], dtype=float).reshape(-1)
r_tracker = np.asarray(hold1_tracker_results["returns"]["tracker"], dtype=float).reshape(-1)
active = np.asarray(hold1_tracker_results["returns"]["active"], dtype=float).reshape(-1)

rolling_te20 = np.asarray(
    hold1_tracker_results["series"]["rolling_te20"],
    dtype=float,
).reshape(-1)

cum_active = np.asarray(
    hold1_tracker_results["series"]["cum_active"],
    dtype=float,
).reshape(-1)

if len(dates) != r_index.size:
    raise ValueError("Length mismatch between dates and Hold-1 return series.")


# ---------------------------------------------------------------
# Useful transformations
# ---------------------------------------------------------------

cum_index = np.cumprod(1.0 + r_index)
cum_tracker = np.cumprod(1.0 + r_tracker)

# Normalize both to one at the first hold date.
cum_index = cum_index / cum_index[0]
cum_tracker = cum_tracker / cum_tracker[0]

daily_abs_te_bp = 1e4 * np.abs(active)
rolling_te20_bp = 1e4 * rolling_te20

te_bp = float(hold1_tracker_results["scalars"]["TE_bp"])


# ---------------------------------------------------------------
# 1) Cumulative return overlay
# ---------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7.2, 3.6))

ax.plot(dates, cum_index, lw=1.2, label="Index")
ax.plot(dates, cum_tracker, lw=1.1, label=TRACKER_LABEL)

ax.set_xlabel("Date")
ax.set_ylabel("Cumulative value")
ax.set_title("Hold-1: cumulative realised returns")
ax.legend(frameon=False)

sparsify_date_axis(ax, maxticks=6)

fig.tight_layout()
fig.savefig(H1_DIR / "hold1_cumret_overlay_tracker.png", dpi=160)
plt.close(fig)


# ---------------------------------------------------------------
# 2) Daily absolute active return in bp
# ---------------------------------------------------------------

ymax_daily = max(5.0, float(np.nanpercentile(daily_abs_te_bp, 99) * 1.10))
ymax_daily = min(ymax_daily, 30.0)

fig, ax = plt.subplots(figsize=(7.2, 3.6))

ax.plot(dates, daily_abs_te_bp, lw=0.9, label=TRACKER_LABEL)
ax.axhline(
    te_bp,
    lw=0.8,
    linestyle="--",
    label=f"Hold TE RMSE: {te_bp:.2f} bp",
)

ax.set_ylim(0.0, ymax_daily)
ax.set_xlabel("Date")
ax.set_ylabel("Daily |active return| (bp)")
ax.set_title("Hold-1: daily realised tracking error")
ax.legend(frameon=False)

sparsify_date_axis(ax, maxticks=6)

fig.tight_layout()
fig.savefig(H1_DIR / "hold1_daily_abs_te_tracker_bp.png", dpi=160)
plt.close(fig)


# ---------------------------------------------------------------
# 3) 20-day rolling RMSE TE in bp
# ---------------------------------------------------------------

valid_roll = ~np.isnan(rolling_te20_bp)

if np.any(valid_roll):
    ymax_roll = max(5.0, float(np.nanpercentile(rolling_te20_bp[valid_roll], 95) * 1.10))
    ymax_roll = min(ymax_roll, 30.0)
else:
    ymax_roll = 5.0

fig, ax = plt.subplots(figsize=(7.2, 3.6))

ax.plot(
    dates,
    rolling_te20_bp,
    lw=1.1,
    label=f"{TRACKER_LABEL}, 20-day RMSE",
)

ax.set_ylim(0.0, ymax_roll)
ax.set_xlabel("Date")
ax.set_ylabel("TE, 20-day RMSE (bp)")
ax.set_title("Hold-1: rolling realised tracking error")
ax.legend(frameon=False)

# This is the plot where dense labels were becoming unreadable.
sparsify_date_axis(ax, maxticks=5)

fig.tight_layout()
fig.savefig(H1_DIR / "hold1_rolling_te20_tracker_bp.png", dpi=160)
plt.close(fig)


# ---------------------------------------------------------------
# 4) Cumulative active return
# ---------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7.2, 3.6))

ax.plot(dates, cum_active, lw=1.0, label=TRACKER_LABEL)

ax.axhline(0.0, lw=0.8, linestyle="--")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative active return")
ax.set_title("Hold-1: cumulative realised active return")
ax.legend(frameon=False)

sparsify_date_axis(ax, maxticks=6)

fig.tight_layout()
fig.savefig(H1_DIR / "hold1_cumulative_active_tracker.png", dpi=160)
plt.close(fig)


# ---------------------------------------------------------------
# 5) Daily active-return histogram
# ---------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6.4, 3.4))

ax.hist(1e4 * active, bins=40)

ax.set_xlabel("Daily active return (bp)")
ax.set_ylabel("Frequency")
ax.set_title("Hold-1: realised active-return histogram")

fig.tight_layout()
fig.savefig(H1_DIR / "hold1_active_hist_tracker_bp.png", dpi=160)
plt.close(fig)


print("\n[Block 12B] Saved Hold-1 figures:")
for fname in [
    "hold1_cumret_overlay_tracker.png",
    "hold1_daily_abs_te_tracker_bp.png",
    "hold1_rolling_te20_tracker_bp.png",
    "hold1_cumulative_active_tracker.png",
    "hold1_active_hist_tracker_bp.png",
]:
    print(f" - {(H1_DIR / fname).resolve()}")

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — tracker cache export
# Export labelled initial Tracker cache for sequential notebook
#
# Purpose:
#   Save the locked FIT-1 Tracker produced by Notebook 1 so that
#   Notebook 2 can start directly from Hold 1.  If APPLY_INITIAL_FLOOR=True,
#   the exported Tracker is the one-off floor-adjusted portfolio.
#
# Filename convention, default:
#   initial_tracker_fit01_nnz150_c25_floor005_53names.npz
#
# where:
#   fit01      = fitting-window number,
#   nnz150     = construction-stage target effective MAP support,
#   c25        = selected effective-variance scale c,
#   floor005   = one-off 0.5% implementability floor applied,
#   53names    = implemented support size after the one-off floor.
#
# The metadata records both the pre-floor posterior-selected support and
# the post-floor implemented support.
# ===============================================================

from pathlib import Path
import json
import re
import sys
import numpy as np
import pandas as pd


# ---------------------------------------------------------------
# 0) Output location
# ---------------------------------------------------------------

TRACKER_CACHE_DIR = Path(OUTPUT_DIR) / "tracker_cache"

TRACKER_CACHE_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------
# 1) Filename convention switches
# ---------------------------------------------------------------

# If False, label uses the target effective MAP nnz, e.g. nnz150.
# If True, label uses the realised selected-row MAP effective nnz,
# e.g. mapnnz152.
USE_ACTUAL_MAP_NNZ_IN_FILENAME = False

# Keep a generic copy as well? Useful while developing, but set to False
# once you have many variants and want to avoid accidental overwrites.
ALSO_SAVE_GENERIC_INITIAL_TRACKER = False


# ---------------------------------------------------------------
# 2) Required object checks
# ---------------------------------------------------------------

required_cache_objects = [
    "w_locked_fit",
    "S_locked_idx_fit",
    "refit_selected_support_fit",
    "selected_support_fit",
    "hold1_tracker_results",
    "idx_name",
    "df_ret",
    "rolling_windows",
]

missing_cache_objects = [
    name for name in required_cache_objects
    if name not in globals()
]

if missing_cache_objects:
    raise RuntimeError(
        "[Tracker cache export] Cannot export Tracker cache. "
        f"Missing required objects: {missing_cache_objects}"
    )


# ---------------------------------------------------------------
# 3) Small robust helpers
# ---------------------------------------------------------------

def _safe_float(x, default=np.nan):
    """
    Convert x to float, returning default if this fails.
    """
    try:
        return float(x)
    except Exception:
        return default


def _safe_int(x, default=None):
    """
    Convert x to int, returning default if this fails.
    """
    try:
        return int(round(float(x)))
    except Exception:
        return default


def _sanitize_label_piece(s: str) -> str:
    """
    Make a string safe for filenames.
    """
    s = str(s)
    s = s.replace(".", "p")
    s = re.sub(r"[^A-Za-z0-9_\-]+", "", s)
    return s


def _format_c_for_label(c_value: float) -> str:
    """
    Format c for a compact filename label.

    Examples
    --------
    25.0  -> c25
    10.0  -> c10
    2.5   -> c2p5
    """
    c_value = float(c_value)

    if abs(c_value - round(c_value)) < 1e-12:
        return f"c{int(round(c_value))}"

    return "c" + _sanitize_label_piece(f"{c_value:g}")


def _find_selected_grid_row():
    """
    Try to locate the selected FIT-grid row from common notebook objects.
    """
    candidates = []

    # Current grid object used in our FIT-1 experiments.
    for obj_name in [
        "fit_grid_smoke",
        "fit_grid_out",
        "fit_grid_fit1",
        "fit_grid_results",
    ]:
        if obj_name in globals() and isinstance(globals()[obj_name], dict):
            candidates.append(globals()[obj_name])

    for obj in candidates:
        if "selected_summary" in obj:
            ss = obj["selected_summary"]

            if isinstance(ss, pd.DataFrame) and len(ss) > 0:
                return ss.iloc[0].to_dict()

            if isinstance(ss, dict):
                return ss

        if "by_window" in obj:
            try:
                bw1 = obj["by_window"][1]

                if isinstance(bw1, dict) and "selected_row" in bw1:
                    return dict(bw1["selected_row"])
            except Exception:
                pass

    # Fallback: some older cells may expose the selected row directly.
    for row_name in [
        "selected_fit_grid_row",
        "selected_row_fit",
        "fit_selected_row",
    ]:
        if row_name in globals():
            row = globals()[row_name]

            if isinstance(row, pd.Series):
                return row.to_dict()

            if isinstance(row, dict):
                return dict(row)

    return {}


def _resolve_filename_metadata():
    """
    Resolve c_star, pre-gating MAP nnz information, and final support size.
    """
    selected_row = _find_selected_grid_row()

    fit_k = int(
        refit_selected_support_fit.get(
            "fit_k",
            selected_support_fit.get("fit_k", 1),
        )
    )

    c_star = _safe_float(
        selected_row.get(
            "c_star",
            selected_row.get("c", refit_selected_support_fit.get("c_star", np.nan)),
        )
    )

    # Prefer explicit target if available.
    nnz_target = _safe_int(
        selected_row.get(
            "nnz_target",
            globals().get("initial_nnz_target", np.nan),
        ),
        default=None,
    )

    # Actual pre-gating MAP effective nnz.
    map_nnz_eff = _safe_int(
        selected_row.get(
            "nnz_eff",
            refit_selected_support_fit.get("map_nnz_eff", np.nan),
        ),
        default=None,
    )

    # Raw MAP nonzeros if available.
    map_nnz_raw = _safe_int(
        selected_row.get(
            "nnz_raw",
            refit_selected_support_fit.get("map_nnz_raw", np.nan),
        ),
        default=None,
    )

    # Record the posterior-selected/refitted support before the optional
    # one-off floor.  This is metadata only; it is not included in the
    # cache filename because the filename is driven by the experimental
    # construction choices: nnz target (or actual MAP nnz) and selected c-star.
    S_for_metadata = np.asarray(
        globals().get("S_locked_idx_pre_floor", S_locked_idx_fit),
        dtype=int,
    ).reshape(-1)
    final_names = int(S_for_metadata.size)

    if not np.isfinite(c_star):
        raise RuntimeError(
            "[Tracker cache export] Could not resolve selected c_star for filename. "
            "Expected fit_grid_smoke['selected_summary'] or equivalent."
        )

    if USE_ACTUAL_MAP_NNZ_IN_FILENAME:
        if map_nnz_eff is None:
            raise RuntimeError(
                "[Tracker cache export] USE_ACTUAL_MAP_NNZ_IN_FILENAME=True but "
                "could not resolve selected MAP nnz_eff."
            )

        nnz_piece = f"mapnnz{map_nnz_eff}"
    else:
        if nnz_target is not None:
            nnz_piece = f"nnz{nnz_target}"
        elif map_nnz_eff is not None:
            nnz_piece = f"mapnnz{map_nnz_eff}"
        else:
            raise RuntimeError(
                "[Tracker cache export] Could not resolve nnz_target or map nnz_eff "
                "for filename."
            )

    c_piece = _format_c_for_label(c_star)

    cache_label = (
        f"fit{fit_k:02d}_"
        f"{nnz_piece}_"
        f"{c_piece}"
    )

    return {
        "selected_row": selected_row,
        "fit_k": fit_k,
        "c_star": float(c_star),
        "nnz_target": nnz_target,
        "map_nnz_eff": map_nnz_eff,
        "map_nnz_raw": map_nnz_raw,
        "final_names": final_names,
        "nnz_piece": nnz_piece,
        "c_piece": c_piece,
        "cache_label": cache_label,
    }


filename_meta = _resolve_filename_metadata()

# Make the default exported cache label explicit about the implemented support.
_floor_applied_for_label = bool(globals().get("INITIAL_FLOOR_APPLIED_TO_LOCKED_TRACKER", False))
_floor_value_for_label = float(globals().get("INITIAL_FLOOR_VALUE_LOCKED_TRACKER", np.nan))
_post_floor_names_for_label = int(np.asarray(S_locked_idx_fit).size)
if _floor_applied_for_label and np.isfinite(_floor_value_for_label):
    # Compact paper-style floor tag: 0.005 -> floor005.
    # This matches the intended maintenance-notebook default cache path.
    _floor_tag = f"{int(round(1000 * _floor_value_for_label)):03d}"
    filename_meta["cache_label"] = f"{filename_meta['cache_label']}_floor{_floor_tag}_{_post_floor_names_for_label}names"
else:
    filename_meta["cache_label"] = f"{filename_meta['cache_label']}_{_post_floor_names_for_label}names"

TRACKER_CACHE_STEM = f"initial_tracker_{filename_meta['cache_label']}"

TRACKER_CACHE_PATH = TRACKER_CACHE_DIR / f"{TRACKER_CACHE_STEM}.npz"
TRACKER_SUMMARY_CSV = TRACKER_CACHE_DIR / f"{TRACKER_CACHE_STEM}_weights.csv"
TRACKER_REFIT_WEIGHTS_CSV = TRACKER_CACHE_DIR / f"{TRACKER_CACHE_STEM}_refit_weights.csv"
TRACKER_HOLD1_SUMMARY_CSV = TRACKER_CACHE_DIR / f"{TRACKER_CACHE_STEM}_hold1_summary.csv"
TRACKER_FLOOR_ACTIONS_CSV = TRACKER_CACHE_DIR / f"{TRACKER_CACHE_STEM}_initial_floor_actions.csv"
TRACKER_METADATA_JSON = TRACKER_CACHE_DIR / f"{TRACKER_CACHE_STEM}_metadata.json"


# ---------------------------------------------------------------
# 4) Resolve core arrays and metadata
# ---------------------------------------------------------------

w_locked_cache = np.asarray(w_locked_fit, dtype=float).reshape(-1)
S_locked_cache = np.asarray(S_locked_idx_fit, dtype=int).reshape(-1)

p_cache = w_locked_cache.size

asset_cols_cache = [
    str(c) for c in df_ret.columns
    if str(c) != str(idx_name)
]

if len(asset_cols_cache) != p_cache:
    raise ValueError(
        "Asset-column length does not match w_locked_fit length. "
        f"len(asset_cols_cache)={len(asset_cols_cache)}, p={p_cache}."
    )

first_window_cache = rolling_windows[0]

# Main FIT TE target for Notebook 2 hit-rate reporting.
initial_fit_te_target = float(
    refit_selected_support_fit["te_fit"]["refit_after_simplex"]
)

fit_te_map_cache = float(refit_selected_support_fit["te_fit"]["map"])
fit_te_refit_before_cache = float(
    refit_selected_support_fit["te_fit"]["refit_before_simplex"]
)
fit_te_refit_after_cache = float(
    refit_selected_support_fit["te_fit"]["refit_after_simplex"]
)

hold1_te_cache = float(hold1_tracker_results["scalars"]["TE_RMSE"])
hold1_te_bp_cache = float(hold1_tracker_results["scalars"]["TE_bp"])

selected_support_rule_cache = str(
    selected_support_fit.get(
        "candidate_name",
        refit_selected_support_fit.get("candidate_name", "unknown"),
    )
)

selected_eps_cache = float(
    selected_support_fit.get(
        "eps",
        refit_selected_support_fit.get("eps", np.nan),
    )
)

selected_pi_pos_cache = float(
    selected_support_fit.get(
        "pi_pos",
        refit_selected_support_fit.get("pi_pos", np.nan),
    )
)



# ---------------------------------------------------------------
# 4b) One-off floor metadata, if Block 11B has been run
# ---------------------------------------------------------------

w_pre_floor_cache = np.asarray(
    globals().get("w_locked_fit_pre_floor", w_locked_cache),
    dtype=float,
).reshape(-1)

if w_pre_floor_cache.size != p_cache:
    raise ValueError(
        "w_locked_fit_pre_floor has incompatible length: "
        f"{w_pre_floor_cache.size} versus p_cache={p_cache}."
    )

S_pre_floor_cache = np.asarray(
    globals().get("S_locked_idx_pre_floor", S_locked_cache),
    dtype=int,
).reshape(-1)

initial_floor_actions_cache = globals().get("initial_floor_actions_fit", pd.DataFrame())
if not isinstance(initial_floor_actions_cache, pd.DataFrame):
    initial_floor_actions_cache = pd.DataFrame(initial_floor_actions_cache)

initial_floor_diagnostics_cache = globals().get("initial_floor_diagnostics_fit", {})
if not isinstance(initial_floor_diagnostics_cache, dict):
    initial_floor_diagnostics_cache = {}

initial_floor_applied_cache = bool(
    globals().get("INITIAL_FLOOR_APPLIED_TO_LOCKED_TRACKER", False)
)
initial_floor_value_cache = _safe_float(
    globals().get("INITIAL_FLOOR_VALUE_LOCKED_TRACKER", np.nan)
)
initial_floor_realloc_cache = str(
    globals().get("INITIAL_FLOOR_REALLOC_LOCKED_TRACKER", "none")
)
initial_floor_turnover_cache = float(
    0.5 * np.sum(np.abs(w_locked_cache - w_pre_floor_cache))
)
initial_floor_mass_pruned_cache = _safe_float(
    initial_floor_diagnostics_cache.get("mass_pruned", np.nan)
)

# ---------------------------------------------------------------
# 5) Build human-readable final-weight table
# ---------------------------------------------------------------

df_initial_tracker_weights = pd.DataFrame(
    {
        "j": np.arange(p_cache, dtype=int),
        "ticker": asset_cols_cache,
        "w_locked_fit": w_locked_cache,
        "w_before_oneoff_floor_fit": w_pre_floor_cache,
        "in_locked_support": np.isin(np.arange(p_cache), S_locked_cache),
        "in_support_before_oneoff_floor": np.isin(np.arange(p_cache), S_pre_floor_cache),
    }
)

df_initial_tracker_weights = (
    df_initial_tracker_weights
    .assign(abs_w_locked_fit=lambda d: np.abs(d["w_locked_fit"]))
    .sort_values("abs_w_locked_fit", ascending=False)
    .reset_index(drop=True)
)

df_initial_tracker_weights.insert(
    0,
    "rank_abs_weight",
    np.arange(1, len(df_initial_tracker_weights) + 1),
)


# ---------------------------------------------------------------
# 6) Refit-weight table if available
# ---------------------------------------------------------------

df_refit_weights_to_save = None

if (
    "refit_reporting_driver_fit" in globals()
    and isinstance(refit_reporting_driver_fit, dict)
    and "df_refit_weights" in refit_reporting_driver_fit
):
    df_refit_weights_to_save = refit_reporting_driver_fit["df_refit_weights"].copy()
elif (
    "df_refit_weights_fit" in globals()
    and isinstance(df_refit_weights_fit, pd.DataFrame)
):
    df_refit_weights_to_save = df_refit_weights_fit.copy()
elif (
    isinstance(refit_selected_support_fit, dict)
    and "df_refit_weights" in refit_selected_support_fit
    and isinstance(refit_selected_support_fit["df_refit_weights"], pd.DataFrame)
):
    df_refit_weights_to_save = refit_selected_support_fit["df_refit_weights"].copy()


# ---------------------------------------------------------------
# 7) Build metadata dictionary
# ---------------------------------------------------------------

tracker_cache_metadata = {
    "cache_name": TRACKER_CACHE_STEM,
    "cache_label": filename_meta["cache_label"],
    "description": (
        "Initial locked Tracker constructed on FIT-1 for the ASOC "
        "Bayesian Index Tracking sequential rebalancing notebook."
    ),

    # Reproducibility metadata.
    "run_mode": str(RUN_MODE),
    "run_production": bool(RUN_PRODUCTION),
    "random_seed": int(RANDOM_SEED),
    "python_version": sys.version,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "numerical_settings": {
        "construction_sapg_n_iter": int(CONSTRUCTION_SAPG_N_ITER),
        "construction_sapg_burn_pr": int(CONSTRUCTION_SAPG_BURN_PR),
        "construction_fista_max_iter": int(CONSTRUCTION_FISTA_MAX_ITER),
        "construction_fista_tol": float(CONSTRUCTION_FISTA_TOL),
        "mala_tune_steps": int(MALA_TUNE_STEPS),
        "construction_mala_n_burn": int(CONSTRUCTION_MALA_N_BURN),
        "construction_mala_n_iter": int(CONSTRUCTION_MALA_N_ITER),
        "construction_mala_thin": int(CONSTRUCTION_MALA_THIN),
        "construction_ess_min_s": float(CONSTRUCTION_ESS_MIN_S),
        "construction_ess_min_te": float(CONSTRUCTION_ESS_MIN_TE),
        "construction_max_lag_ess": int(CONSTRUCTION_MAX_LAG_ESS),
    },

    # Filename-driving metadata.
    "fit_k": int(filename_meta["fit_k"]),
    "c_star": float(filename_meta["c_star"]),
    "c_piece": filename_meta["c_piece"],
    "nnz_piece": filename_meta["nnz_piece"],
    "nnz_target": (
        None if filename_meta["nnz_target"] is None
        else int(filename_meta["nnz_target"])
    ),
    "map_nnz_eff": (
        None if filename_meta["map_nnz_eff"] is None
        else int(filename_meta["map_nnz_eff"])
    ),
    "map_nnz_raw": (
        None if filename_meta["map_nnz_raw"] is None
        else int(filename_meta["map_nnz_raw"])
    ),
    "final_names": int(filename_meta["final_names"]),
    "support_size_after_oneoff_floor": int(S_locked_cache.size),
    "use_actual_map_nnz_in_filename": bool(USE_ACTUAL_MAP_NNZ_IN_FILENAME),

    # Data/source conventions.
    "path_merged": str(globals().get("PATH_MERGED", "")),
    "idx_name": str(idx_name),
    "n_assets": int(p_cache),
    "asset_cols": asset_cols_cache,

    # Window metadata.
    "fit_start": str(first_window_cache["fit_start"]),
    "fit_end": str(first_window_cache["fit_end"]),
    "hold1_start": str(first_window_cache["hold_start"]),
    "hold1_end": str(first_window_cache["hold_end"]),

    # Support rule.
    "selected_support_rule": selected_support_rule_cache,
    "selected_eps": selected_eps_cache,
    "selected_pi_pos": selected_pi_pos_cache,
    "support_size": int(S_locked_cache.size),
    "support_size_before_oneoff_floor": int(S_pre_floor_cache.size),

    # One-off floor/export convention.
    "initial_floor_applied_to_cache": bool(initial_floor_applied_cache),
    "initial_floor_value": (
        None if not np.isfinite(initial_floor_value_cache)
        else float(initial_floor_value_cache)
    ),
    "initial_floor_realloc": initial_floor_realloc_cache,
    "initial_floor_turnover_from_refit": float(initial_floor_turnover_cache),
    "initial_floor_mass_pruned": (
        None if not np.isfinite(initial_floor_mass_pruned_cache)
        else float(initial_floor_mass_pruned_cache)
    ),

    # Portfolio structure.
    "sum_w_locked_fit_before_oneoff_floor": float(w_pre_floor_cache.sum()),
    "sum_w_locked_fit": float(w_locked_cache.sum()),
    "nnz_locked_fit": int(np.count_nonzero(np.abs(w_locked_cache) > 1e-12)),
    "n_shorts_locked_fit": int(np.count_nonzero(w_locked_cache < -1e-12)),
    "min_w_locked_fit": float(np.min(w_locked_cache)),
    "max_w_locked_fit": float(np.max(w_locked_cache)),
    "L1_locked_fit": float(np.sum(np.abs(w_locked_cache))),

    # FIT TE.
    "initial_fit_te_target": initial_fit_te_target,
    "fit_te_map": fit_te_map_cache,
    "fit_te_refit_before_simplex": fit_te_refit_before_cache,
    "fit_te_refit_after_simplex": fit_te_refit_after_cache,

    # Hold-1 realised diagnostics.
    "hold1_te_rmse": hold1_te_cache,
    "hold1_te_bp": hold1_te_bp_cache,
    "hold1_mae_bp": float(hold1_tracker_results["scalars"]["MAE_bp"]),
    "hold1_mean_active_bp": float(hold1_tracker_results["scalars"]["mean_active_bp"]),
    "hold1_std_active_bp": float(hold1_tracker_results["scalars"]["std_active_bp"]),
    "hold1_hit_rate_vs_fit_te": float(
        hold1_tracker_results["scalars"]["hit_rate_vs_FIT_TE"]
    ),
}


# ---------------------------------------------------------------
# 8) Save compact NPZ cache
# ---------------------------------------------------------------

np.savez(
    TRACKER_CACHE_PATH,

    # Core arrays for Notebook 2.
    w_locked_fit=w_locked_cache,
    S_locked_idx_fit=S_locked_cache,
    w_locked_fit_pre_floor=w_pre_floor_cache,
    S_locked_idx_pre_floor=S_pre_floor_cache,
    asset_cols=np.asarray(asset_cols_cache, dtype=object),
    idx_name=np.asarray(str(idx_name), dtype=object),

    # Main reporting scalar.
    initial_fit_te_target=np.asarray(initial_fit_te_target, dtype=float),

    # Support-rule metadata.
    selected_support_rule_fit=np.asarray(selected_support_rule_cache, dtype=object),
    selected_eps_fit=np.asarray(selected_eps_cache, dtype=float),
    selected_pi_pos_fit=np.asarray(selected_pi_pos_cache, dtype=float),

    # Filename-driving metadata.
    tracker_cache_label=np.asarray(filename_meta["cache_label"], dtype=object),
    c_star=np.asarray(filename_meta["c_star"], dtype=float),
    nnz_target=np.asarray(
        np.nan if filename_meta["nnz_target"] is None
        else filename_meta["nnz_target"],
        dtype=float,
    ),
    map_nnz_eff=np.asarray(
        np.nan if filename_meta["map_nnz_eff"] is None
        else filename_meta["map_nnz_eff"],
        dtype=float,
    ),
    final_names=np.asarray(filename_meta["final_names"], dtype=int),
    post_floor_names=np.asarray(S_locked_cache.size, dtype=int),
    initial_floor_applied=np.asarray(initial_floor_applied_cache, dtype=bool),
    initial_floor_value=np.asarray(initial_floor_value_cache, dtype=float),
    initial_floor_turnover_from_refit=np.asarray(initial_floor_turnover_cache, dtype=float),
    initial_floor_mass_pruned=np.asarray(initial_floor_mass_pruned_cache, dtype=float),

    # Window metadata.
    fit_start=np.asarray(str(first_window_cache["fit_start"]), dtype=object),
    fit_end=np.asarray(str(first_window_cache["fit_end"]), dtype=object),
    hold1_start=np.asarray(str(first_window_cache["hold_start"]), dtype=object),
    hold1_end=np.asarray(str(first_window_cache["hold_end"]), dtype=object),

    # JSON metadata as a string.
    metadata_json=np.asarray(json.dumps(tracker_cache_metadata), dtype=object),
)


# ---------------------------------------------------------------
# 9) Save readable companion files
# ---------------------------------------------------------------

df_initial_tracker_weights.to_csv(
    TRACKER_SUMMARY_CSV,
    index=False,
)

if df_refit_weights_to_save is not None:
    df_refit_weights_to_save.to_csv(
        TRACKER_REFIT_WEIGHTS_CSV,
        index=False,
    )

hold1_tracker_results["df_summary"].to_csv(
    TRACKER_HOLD1_SUMMARY_CSV,
    index=False,
)

initial_floor_actions_cache.to_csv(
    TRACKER_FLOOR_ACTIONS_CSV,
    index=False,
)

with open(TRACKER_METADATA_JSON, "w") as f:
    json.dump(tracker_cache_metadata, f, indent=2)

# Convenience pointer for Notebook 2.
LATEST_TRACKER_CACHE_PATH_TXT = TRACKER_CACHE_DIR / "latest_tracker_cache_path.txt"
LATEST_TRACKER_CACHE_PATH_TXT.write_text(str(TRACKER_CACHE_PATH), encoding="utf-8")


# Optional generic copy for development.
if ALSO_SAVE_GENERIC_INITIAL_TRACKER:
    generic_npz = TRACKER_CACHE_DIR / "initial_tracker_fit01.npz"
    generic_weights = TRACKER_CACHE_DIR / "initial_tracker_fit01_weights.csv"
    generic_refit = TRACKER_CACHE_DIR / "initial_tracker_fit01_refit_weights.csv"
    generic_hold1 = TRACKER_CACHE_DIR / "initial_tracker_fit01_hold1_summary.csv"
    generic_metadata = TRACKER_CACHE_DIR / "initial_tracker_fit01_metadata.json"
    generic_floor_actions = TRACKER_CACHE_DIR / "initial_tracker_fit01_initial_floor_actions.csv"

    np.savez(
        generic_npz,
        w_locked_fit=w_locked_cache,
        S_locked_idx_fit=S_locked_cache,
        asset_cols=np.asarray(asset_cols_cache, dtype=object),
        idx_name=np.asarray(str(idx_name), dtype=object),
        initial_fit_te_target=np.asarray(initial_fit_te_target, dtype=float),
        selected_support_rule_fit=np.asarray(selected_support_rule_cache, dtype=object),
        selected_eps_fit=np.asarray(selected_eps_cache, dtype=float),
        selected_pi_pos_fit=np.asarray(selected_pi_pos_cache, dtype=float),
        tracker_cache_label=np.asarray(filename_meta["cache_label"], dtype=object),
        c_star=np.asarray(filename_meta["c_star"], dtype=float),
        nnz_target=np.asarray(
            np.nan if filename_meta["nnz_target"] is None
            else filename_meta["nnz_target"],
            dtype=float,
        ),
        map_nnz_eff=np.asarray(
            np.nan if filename_meta["map_nnz_eff"] is None
            else filename_meta["map_nnz_eff"],
            dtype=float,
        ),
        final_names=np.asarray(filename_meta["final_names"], dtype=int),
    post_floor_names=np.asarray(S_locked_cache.size, dtype=int),
    initial_floor_applied=np.asarray(initial_floor_applied_cache, dtype=bool),
    initial_floor_value=np.asarray(initial_floor_value_cache, dtype=float),
    initial_floor_turnover_from_refit=np.asarray(initial_floor_turnover_cache, dtype=float),
    initial_floor_mass_pruned=np.asarray(initial_floor_mass_pruned_cache, dtype=float),
        fit_start=np.asarray(str(first_window_cache["fit_start"]), dtype=object),
        fit_end=np.asarray(str(first_window_cache["fit_end"]), dtype=object),
        hold1_start=np.asarray(str(first_window_cache["hold_start"]), dtype=object),
        hold1_end=np.asarray(str(first_window_cache["hold_end"]), dtype=object),
        metadata_json=np.asarray(json.dumps(tracker_cache_metadata), dtype=object),
    )

    df_initial_tracker_weights.to_csv(generic_weights, index=False)

    if df_refit_weights_to_save is not None:
        df_refit_weights_to_save.to_csv(generic_refit, index=False)

    hold1_tracker_results["df_summary"].to_csv(generic_hold1, index=False)
    initial_floor_actions_cache.to_csv(generic_floor_actions, index=False)

    with open(generic_metadata, "w") as f:
        json.dump(tracker_cache_metadata, f, indent=2)


# ---------------------------------------------------------------
# 10) Print export summary
# ---------------------------------------------------------------

print("\n[Tracker cache export] Initial Tracker cache exported.")
print(f"Cache label:     {filename_meta['cache_label']}")
print(f"NPZ cache:       {TRACKER_CACHE_PATH}")
print(f"Weights CSV:     {TRACKER_SUMMARY_CSV}")

if df_refit_weights_to_save is not None:
    print(f"Refit CSV:       {TRACKER_REFIT_WEIGHTS_CSV}")
else:
    print("Refit CSV:       not saved; df_refit_weights table was not found")

print(f"Hold-1 CSV:      {TRACKER_HOLD1_SUMMARY_CSV}")
print(f"Floor actions:   {TRACKER_FLOOR_ACTIONS_CSV}")
print(f"Metadata JSON:   {TRACKER_METADATA_JSON}")
print(f"Latest pointer:  {LATEST_TRACKER_CACHE_PATH_TXT}")

if ALSO_SAVE_GENERIC_INITIAL_TRACKER:
    print("\n[Tracker cache export] Generic development copies also saved as initial_tracker_fit01.*")

print("\n[Initial Tracker cache summary]")
print(f"selected_support_rule = {selected_support_rule_cache}")
print(f"eps = {selected_eps_cache:.1e}")
print(f"pi_pos = {selected_pi_pos_cache:.2f}")
print(f"selected c_star = {filename_meta['c_star']:.6g}")
print(f"filename nnz piece = {filename_meta['nnz_piece']}")

if filename_meta["map_nnz_eff"] is not None:
    print(f"selected MAP nnz_eff = {filename_meta['map_nnz_eff']}")

if filename_meta["nnz_target"] is not None:
    print(f"selected/grid nnz_target = {filename_meta['nnz_target']}")

print(f"|S_locked recorded in metadata| = {filename_meta['final_names']}")
print(f"|S_locked before one-off floor| = {S_pre_floor_cache.size}")
print(f"|S_locked after one-off floor|  = {S_locked_cache.size}")
print(f"initial floor applied to cache = {initial_floor_applied_cache}")
print(f"initial floor turnover from refit = {initial_floor_turnover_cache:.6e}")
print(f"sum(w_locked_fit) = {w_locked_cache.sum():+.12f}")
print(f"nnz(w_locked_fit) = {np.count_nonzero(np.abs(w_locked_cache) > 1e-12)}")
print(f"shorts(w_locked_fit) = {np.count_nonzero(w_locked_cache < -1e-12)}")
print(
    f"initial_fit_te_target = {initial_fit_te_target:.6e} "
    f"({1e4 * initial_fit_te_target:.3f} bp)"
)
print(
    f"Hold-1 realised TE = {hold1_te_cache:.6e} "
    f"({hold1_te_bp_cache:.3f} bp)"
)

In [ ]:
# ===============================================================
# GitHub construction audit exports (CSV/JSON only; no LaTeX)
# ===============================================================
from pathlib import Path
import json
import sys

CONSTRUCTION_AUDIT_DIR = Path(OUTPUT_DIR) / "construction_audit"
CONSTRUCTION_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# Construction c-grid rows and selected row.
fit_grid_out["by_window"][fit_k_current]["summary"].to_csv(
    CONSTRUCTION_AUDIT_DIR / "construction_fit01_cgrid_summary.csv",
    index=False,
)
fit_grid_out["selected_summary"].to_csv(
    CONSTRUCTION_AUDIT_DIR / "construction_selected_summary.csv",
    index=False,
)

# Support sensitivity and selected support weights, when available.
if "df_support_frontier_fit" in globals():
    df_support_frontier_fit.to_csv(
        CONSTRUCTION_AUDIT_DIR / "practical_positive_support_frontier.csv",
        index=False,
    )
if "df_refit_weights_fit" in globals():
    df_refit_weights_fit.to_csv(
        CONSTRUCTION_AUDIT_DIR / "selected_support_refit_weights.csv",
        index=False,
    )
if "initial_floor_actions_fit" in globals():
    initial_floor_actions_fit.to_csv(
        CONSTRUCTION_AUDIT_DIR / "initial_floor_actions.csv",
        index=False,
    )
if "hold1_tracker_results" in globals() and isinstance(hold1_tracker_results, dict):
    hold1_tracker_results["df_summary"].to_csv(
        CONSTRUCTION_AUDIT_DIR / "hold1_tracker_summary.csv",
        index=False,
    )

manifest = {
    "run_mode": RUN_MODE,
    "run_production": bool(RUN_PRODUCTION),
    "random_seed": int(RANDOM_SEED),
    "python_version": sys.version,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "data_path": str(DATA_PATH),
    "output_root": str(OUTPUT_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "tracker_cache_path": str(globals().get("TRACKER_CACHE_PATH", "")),
    "tracker_metadata_json": str(globals().get("TRACKER_METADATA_JSON", "")),
    "nnz_target": int(NNZ_TARGET),
    "practical_pos_eps": float(PRACTICAL_POS_EPS),
    "practical_pos_prob": float(PRACTICAL_POS_PROB),
    "initial_floor_applied": bool(APPLY_INITIAL_FLOOR),
    "initial_floor_weight": float(INITIAL_FLOOR_WEIGHT),
    "numerical_settings": {
        "construction_sapg_n_iter": int(CONSTRUCTION_SAPG_N_ITER),
        "construction_sapg_burn_pr": int(CONSTRUCTION_SAPG_BURN_PR),
        "construction_fista_max_iter": int(CONSTRUCTION_FISTA_MAX_ITER),
        "construction_mala_n_burn": int(CONSTRUCTION_MALA_N_BURN),
        "construction_mala_n_iter": int(CONSTRUCTION_MALA_N_ITER),
        "construction_mala_thin": int(CONSTRUCTION_MALA_THIN),
    },
}
(CONSTRUCTION_AUDIT_DIR / "construction_audit_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print("\n[GitHub construction audit exports]")
print(f"audit directory = {CONSTRUCTION_AUDIT_DIR.resolve()}")
print(f"tracker cache   = {globals().get('TRACKER_CACHE_PATH', '')}")

In [ ]:
# ===============================================================
# Paper-number validation for production construction runs
# ===============================================================
# This cell is intentionally tolerant: tiny stochastic differences may occur
# when the long MALA run is repeated on a different machine.

EXPECTED_CONSTRUCTION = {
    "c_star": 25.0,
    "support_before_floor": 59,
    "support_after_floor": 53,
    "floor_turnover_pct": 1.806,
    "hold1_te_bp": 17.151,
}

if RUN_PRODUCTION:
    observed = {
        "c_star": float(filename_meta["c_star"]),
        "support_before_floor": int(S_pre_floor_cache.size),
        "support_after_floor": int(S_locked_cache.size),
        "floor_turnover_pct": 100.0 * float(initial_floor_turnover_cache),
        "hold1_te_bp": float(hold1_te_bp_cache),
    }

    tolerances = {
        "c_star": 1e-12,
        "support_before_floor": 0,
        "support_after_floor": 0,
        "floor_turnover_pct": 0.01,
        "hold1_te_bp": 0.05,
    }

    print("\n[Paper-number validation: construction]")
    for key, expected in EXPECTED_CONSTRUCTION.items():
        obs = observed[key]
        tol = tolerances[key]
        ok = abs(float(obs) - float(expected)) <= tol
        status = "OK" if ok else "CHECK"
        print(f"{status:5s} {key:24s} observed={obs} expected={expected} tol={tol}")
else:
    print("[Paper-number validation skipped: RUN_PRODUCTION=False smoke mode]")
